## Connect LLM

In [ ]:
from aipipe import LLM

In [7]:
from pathlib import Path
from dotenv import load_dotenv
from aipipe import LLM

# AIPipe project root
PROJECT_ROOT = Path(r"C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe")

# Load .env
load_dotenv(PROJECT_ROOT / ".env")

# LLM configuration
CONFIG_DIR = PROJECT_ROOT / "config"
LLM_CONFIG = CONFIG_DIR / "llm.yaml"

print("Project Root:", PROJECT_ROOT)
print("Config:", LLM_CONFIG)
print("Config exists:", LLM_CONFIG.exists())

# Load LLM
llm = LLM.from_config(str(LLM_CONFIG))

Project Root: C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe
Config: C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe\config\llm.yaml
Config exists: True


In [ ]:
llm.generate("Hello, world!")

# Q & A Evaluation Agent

In [ ]:
from pathlib import Path
from pydantic import BaseModel, Field


In [ ]:
class AnswerEvaluation(BaseModel):

    question: str

    student_answer: str

    max_marks: float

    marks_obtained: float

    percentage: float

    grade: str

    correctness_score: float

    completeness_score: float

    relevance_score: float

    clarity_score: float

    feedback: str

    strengths: list[str] = Field(
        default_factory=list
    )

    improvements: list[str] = Field(
        default_factory=list
    )

    missing_concepts: list[str] = Field(
        default_factory=list
    )

    justification: str

In [ ]:
ANSWER_EVALUATOR_CONFIG = {

    "name": "answer_evaluator",

    "description": (
        "Evaluates student answers against questions "
        "and assigns marks with detailed feedback."
    ),

    "system_prompt": """
You are an academic answer evaluation agent.

Your task is to evaluate a student's answer against
a given question.

Evaluate the answer based on:

1. Correctness
2. Completeness
3. Relevance
4. Clarity

You must:

- Understand the question before evaluating.
- Identify the important concepts required by the question.
- Analyze the student's answer.
- Check whether the concepts are correct.
- Check whether important concepts are missing.
- Check whether the answer is relevant.
- Check whether the explanation is clear.
- Award marks fairly.
- Do not invent information.
- Do not penalize grammar unless grammar is explicitly
  part of the evaluation criteria.
- Provide constructive feedback.
- Identify strengths.
- Identify improvements.
- Identify missing concepts.
- Explain why the marks were awarded.

Return only the structured evaluation.
""",

    "max_iterations": 5,

    "tools": [],

    "memory_enabled": False,

    "security_enabled": True,
}

ANSWER_EVALUATOR_CONFIG

In [ ]:
from typing import Optional

In [ ]:
def build_evaluation_prompt(
    question: str,
    student_answer: str,
    max_marks: float = 10,
    expected_answer: str | None = None,
):

    reference_section = ""

    if expected_answer:
        reference_section = f"""
REFERENCE / EXPECTED ANSWER:

{expected_answer}

Use this reference answer to evaluate
correctness and completeness.
"""

    prompt = f"""
{ANSWER_EVALUATOR_CONFIG["system_prompt"]}

==================================================
QUESTION
==================================================

{question}

==================================================
STUDENT ANSWER
==================================================

{student_answer}

==================================================
MAXIMUM MARKS
==================================================

{max_marks}

{reference_section}

==================================================
EVALUATION
==================================================

Evaluate the student's answer.

Determine:

- correctness_score
- completeness_score
- relevance_score
- clarity_score
- marks_obtained
- percentage
- grade

Also provide:

- feedback
- strengths
- improvements
- missing_concepts
- justification

The four evaluation scores should be based on:

Correctness: 50%
Completeness: 25%
Relevance: 15%
Clarity: 10%

marks_obtained must be between 0 and {max_marks}.

percentage must be calculated as:

(marks_obtained / max_marks) * 100

Return only the structured result.

Do not return markdown.
Do not return ```json.
Do not add explanations outside the structured result.
"""

    return prompt

In [ ]:
def evaluate_answer(
    question: str,
    student_answer: str,
    max_marks: float = 10,
    expected_answer: Optional[str] = None,
):

    if not question.strip():
        raise ValueError(
            "Question cannot be empty."
        )

    if not student_answer.strip():
        raise ValueError(
            "Student answer cannot be empty."
        )

    if max_marks <= 0:
        raise ValueError(
            "Maximum marks must be greater than zero."
        )

    prompt = build_evaluation_prompt(
        question=question,
        student_answer=student_answer,
        max_marks=max_marks,
        expected_answer=expected_answer,
    )

    result = llm.structured(
        AnswerEvaluation,
        prompt,
    )

    return result

In [ ]:
question = """
Explain the difference between a Python list and a tuple.
Give two differences and one example of each.
"""

student_answer = """
A list is mutable while a tuple is immutable.

Lists use square brackets and tuples use parentheses.

Example of list:
numbers = [1, 2, 3]

Example of tuple:
numbers = (1, 2, 3)
"""

In [ ]:
result = evaluate_answer(
    question=question,
    student_answer=student_answer,
    max_marks=10,
)

print(result)

In [ ]:
result.model_dump()

In [ ]:
question = """
What is inheritance in Python?
Explain inheritance and provide an example.
"""

student_answer = """
Inheritance allows one class to use the properties
and methods of another class.
"""

expected_answer = """
Inheritance is an object-oriented programming concept
where a child class inherits attributes and methods
from a parent class.

Example:

class Animal:
    def speak(self):
        print("Animal sound")

class Dog(Animal):
    pass
"""

In [ ]:
import json

In [ ]:
result = evaluate_answer(
    question=question,
    student_answer=student_answer,
    max_marks=10,
    expected_answer=expected_answer,
)

print(
    json.dumps(
        result.model_dump(),
        indent=2
    )
)

# Resume Agent

In [ ]:
class ResumeAnalysis(BaseModel):

    candidate_name: str

    professional_summary: str

    education: list[str] = Field(
        default_factory=list
    )

    technical_skills: list[str] = Field(
        default_factory=list
    )

    soft_skills: list[str] = Field(
        default_factory=list
    )

    programming_languages: list[str] = Field(
        default_factory=list
    )

    frameworks_and_libraries: list[str] = Field(
        default_factory=list
    )

    databases: list[str] = Field(
        default_factory=list
    )

    tools_and_technologies: list[str] = Field(
        default_factory=list
    )

    certifications: list[str] = Field(
        default_factory=list
    )

    projects: list[str] = Field(
        default_factory=list
    )

    work_experience: list[str] = Field(
        default_factory=list
    )

    total_experience_years: float

    key_strengths: list[str] = Field(
        default_factory=list
    )

    areas_for_improvement: list[str] = Field(
        default_factory=list
    )

    missing_skills: list[str] = Field(
        default_factory=list
    )

    career_roles: list[str] = Field(
        default_factory=list
    )

    career_recommendations: list[str] = Field(
        default_factory=list
    )

    resume_quality_score: float

    feedback: str

In [ ]:
RESUME_ANALYZER_CONFIG = {

    "name": "resume_analyzer",

    "description": (
        "Analyzes student resumes and extracts "
        "skills, education, experience, projects "
        "and career recommendations."
    ),

    "system_prompt": """
You are a professional resume analysis agent.

Your task is to analyze a student's resume and
produce a structured analysis.

Extract information directly from the resume.

Analyze:

- Candidate information
- Professional summary
- Education
- Technical skills
- Programming languages
- Frameworks and libraries
- Databases
- Tools and technologies
- Soft skills
- Certifications
- Projects
- Work experience
- Total experience
- Key strengths
- Areas for improvement
- Missing skills
- Suitable career roles
- Career recommendations

Rules:

1. Do not invent information.
2. If information is not present, return an empty list
   or an appropriate zero value.
3. Distinguish between skills explicitly mentioned in
   the resume and skills inferred from projects.
4. Do not claim that the candidate has a skill unless
   there is evidence in the resume.
5. Career recommendations should be based on the
   skills, education, projects and experience found
   in the resume.
6. Provide constructive feedback.
7. Evaluate the overall quality and completeness of
   the resume.
8. Return only the requested structured output.
"""
}

In [ ]:
def build_resume_prompt(
    resume_text: str,
):

    prompt = f"""
{RESUME_ANALYZER_CONFIG["system_prompt"]}

==================================================
RESUME
==================================================

{resume_text}

==================================================
ANALYSIS
==================================================

Analyze the resume.

Extract the candidate's:

- Name
- Professional summary
- Education
- Technical skills
- Soft skills
- Programming languages
- Frameworks and libraries
- Databases
- Tools and technologies
- Certifications
- Projects
- Work experience
- Total experience

Then determine:

- Key strengths
- Areas for improvement
- Missing skills
- Suitable career roles
- Career recommendations
- Resume quality score
- Overall feedback

The resume quality score should be between 0 and 100.

Do not invent information.

If information is unavailable, use:

[] for lists

0 for numeric values

"" for unavailable text fields.

Return only the structured result.
Do not return markdown.
Do not return ```json.
Do not add explanations outside the structured result.
"""

    return prompt

In [ ]:
def analyze_resume(
    resume_text: str,
) -> ResumeAnalysis:

    if not resume_text or not resume_text.strip():
        raise ValueError(
            "Resume text cannot be empty."
        )

    prompt = build_resume_prompt(
        resume_text
    )

    result = llm.structured(
        ResumeAnalysis,
        prompt,
    )

    return result

In [ ]:
resume_text = """
John Doe

Email: john@example.com

PROFESSIONAL SUMMARY

Computer Science graduate with experience in Python,
machine learning and web development.

EDUCATION

B.Tech in Computer Science Engineering
ABC Engineering College
2024

TECHNICAL SKILLS

Python, Java, SQL, Machine Learning, Deep Learning,
FastAPI, Flask, React, MongoDB, Git

PROJECTS

1. Student Performance Prediction
Developed a machine learning model using Python
and scikit-learn to predict student performance.

2. Resume Analyzer
Built a resume analysis application using Python,
FastAPI and an LLM.

CERTIFICATIONS

Python Programming Certification

EXPERIENCE

Software Development Intern
XYZ Technologies
6 months

Worked on Python APIs and database integration.
"""

In [ ]:
result = analyze_resume(
    resume_text
)

print(result)

In [ ]:
import json

print(
    json.dumps(
        result.model_dump(),
        indent=2
    )
)

In [ ]:
!pip install PyMuPDF python-docx

In [ ]:
import fitz  # PyMuPDF
from docx import Document

In [ ]:
def extract_resume_text(file_path: str) -> str:
    """
    Extract text from PDF, DOCX, or TXT resume.

    Parameters
    ----------
    file_path : str
        Path to the resume file.

    Returns
    -------
    str
        Extracted resume text.
    """

    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"Resume file not found: {path}"
        )

    extension = path.suffix.lower()

    # ------------------------------------------------------
    # TXT
    # ------------------------------------------------------

    if extension == ".txt":

        return path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

    # ------------------------------------------------------
    # PDF
    # ------------------------------------------------------

    elif extension == ".pdf":

        text_parts = []

        with fitz.open(path) as pdf:

            for page in pdf:
                text = page.get_text()

                if text:
                    text_parts.append(text)

        return "\n".join(text_parts).strip()

    # ------------------------------------------------------
    # DOCX
    # ------------------------------------------------------

    elif extension == ".docx":

        document = Document(path)

        paragraphs = []

        for paragraph in document.paragraphs:

            text = paragraph.text.strip()

            if text:
                paragraphs.append(text)

        # Also extract table content
        for table in document.tables:

            for row in table.rows:

                row_text = []

                for cell in row.cells:

                    cell_text = cell.text.strip()

                    if cell_text:
                        row_text.append(cell_text)

                if row_text:
                    paragraphs.append(
                        " | ".join(row_text)
                    )

        return "\n".join(paragraphs).strip()

    # ------------------------------------------------------
    # Unsupported
    # ------------------------------------------------------

    else:

        raise ValueError(
            f"Unsupported resume format: {extension}. "
            "Supported formats are PDF, DOCX and TXT."
        )

In [ ]:
def analyze_resume_file(
    file_path: str,
    save_json: bool = False,
    output_path: str | None = None,
):
    """
    Extract a resume from PDF/DOCX/TXT and analyze it.

    Returns a JSON-compatible dictionary.
    """

    # ---------------------------------------------
    # Extract resume
    # ---------------------------------------------

    resume_text = extract_resume_text(
        file_path
    )

    if not resume_text:
        raise ValueError(
            "No text could be extracted from the resume."
        )

    # ---------------------------------------------
    # Analyze using existing agent
    # ---------------------------------------------

    result = analyze_resume(
        resume_text
    )

    # ---------------------------------------------
    # Convert structured output to dictionary
    # ---------------------------------------------

    result_json = result.model_dump()

    # ---------------------------------------------
    # Save JSON if requested
    # ---------------------------------------------

    if save_json:

        if output_path is None:

            input_path = Path(file_path)

            output_path = str(
                input_path.with_suffix(".json")
            )

        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                result_json,
                file,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"JSON saved to: {output_path}"
        )

    return result_json

In [ ]:
result = analyze_resume_file(
    r"C:\Users\Deepak\Downloads\Deepak Hariharapanda-1.pdf"
)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# 

# Speech to text

In [ ]:
import re


# ============================================================
# CHECK WHETHER TEXT CONTAINS TELUGU SCRIPT
# ============================================================

def contains_telugu_script(text):
    """
    Telugu Unicode range:
        U+0C00 - U+0C7F
    """

    if not text:
        return False

    telugu_chars = re.findall(
        r"[\u0C00-\u0C7F]",
        text
    )

    return len(telugu_chars) > 0


# ============================================================
# CALCULATE TELUGU CHARACTER RATIO
# ============================================================

def telugu_character_ratio(text):
    """
    Returns percentage of characters that belong
    to the Telugu Unicode block.
    """

    if not text:
        return 0.0

    # Remove spaces and punctuation
    meaningful_chars = [
        ch for ch in text
        if ch.isalpha()
    ]

    if not meaningful_chars:
        return 0.0

    telugu_chars = [
        ch for ch in meaningful_chars
        if "\u0C00" <= ch <= "\u0C7F"
    ]

    return len(telugu_chars) / len(meaningful_chars)


# ============================================================
# VERIFY TELUGU WHEN DETECTION SAYS ENGLISH
# ============================================================

def verify_telugu_audio(audio_path):
    """
    Run the Telugu Whisper model as a verification step.

    This is particularly useful for code-mixed speech such as:

        Telugu + English words

    Example:
        "నేను Python programming నేర్చుకుంటున్నాను"
    """

    print("\nRunning Telugu verification...")

    telugu_text = transcribe_telugu(
        audio_path
    )

    ratio = telugu_character_ratio(
        telugu_text
    )

    print(
        "Telugu verification transcription:",
        telugu_text
    )

    print(
        f"Telugu character ratio: {ratio:.2%}"
    )

    return telugu_text, ratio


# ============================================================
# IMPROVED SPEECH-TO-TEXT
# ============================================================

def speech_to_text(audio_path):

    if not audio_path:

        return (
            "No audio recorded.",
            ""
        )

    print("\n" + "=" * 60)
    print("PROCESSING AUDIO")
    print("=" * 60)

    # --------------------------------------------------------
    # STEP 1
    # Initial Whisper language detection
    # --------------------------------------------------------

    code, language, confidence = detect_language(
        audio_path
    )

    print(
        f"\nInitial detection: {language}"
    )

    print(
        f"Confidence: {confidence:.2%}"
    )

    # --------------------------------------------------------
    # STEP 2
    # If English is detected, verify Telugu.
    #
    # This is the important part for:
    #
    # Telugu + English
    # Telugu + technical terms
    # Telugu + programming terms
    # --------------------------------------------------------

    if code == "en":

        print(
            "\nEnglish detected."
            "\nChecking whether speech is actually Telugu "
            "with English words..."
        )

        telugu_text, telugu_ratio = (
            verify_telugu_audio(audio_path)
        )

        # ----------------------------------------------------
        # If Telugu script is clearly present,
        # override English detection.
        # ----------------------------------------------------

        if telugu_ratio >= 0.15:

            print(
                "\nTelugu speech detected "
                "despite English words."
            )

            code = "te"
            language = "Telugu"

            transcript = telugu_text

            print(
                "Final language: Telugu"
            )

            print(
                "Final transcript:",
                transcript
            )

            return (
                f"Language: Telugu\n"
                f"Detection: Telugu verification\n"
                f"Telugu ratio: {telugu_ratio:.2%}",
                transcript
            )

    # --------------------------------------------------------
    # STEP 3
    # Normal model routing
    # --------------------------------------------------------

    if code == "te":

        print(
            "\nUsing Telugu Whisper model..."
        )

        transcript = transcribe_telugu(
            audio_path
        )

    elif code == "hi":

        print(
            "\nUsing Whisper Small for Hindi..."
        )

        transcript = transcribe_multilingual(
            audio_path,
            "hi"
        )

    else:

        print(
            "\nUsing Whisper Small for English..."
        )

        transcript = transcribe_multilingual(
            audio_path,
            "en"
        )

    # --------------------------------------------------------
    # STEP 4
    # Final result
    # --------------------------------------------------------

    print(
        "\nFinal language:",
        language
    )

    print(
        "Final transcript:",
        transcript
    )

    result = (
        f"Language: {language}\n"
        f"Confidence: {confidence:.2%}"
    )

    return (
        result,
        transcript
    )

In [ ]:
# ============================================================
# MULTILINGUAL SPEECH-TO-TEXT
#
# Telugu  -> vasista22/whisper-telugu-small
# Hindi   -> openai/whisper-small
# English -> openai/whisper-small
#
# IMPORTANT:
# If language detection says English but the English
# transcription is not actually English, Telugu model
# is used as a fallback.
# ============================================================

import re
import torch
import librosa
import gradio as gr

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

TELUGU_MODEL = "vasista22/whisper-telugu-small"
MULTILINGUAL_MODEL = "openai/whisper-small"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)

print("=" * 60)
print("MULTILINGUAL SPEECH-TO-TEXT")
print("=" * 60)
print("Device:", DEVICE)


# ============================================================
# 2. LOAD TELUGU MODEL
# ============================================================

print("\nLoading Telugu Whisper model...")

te_processor = AutoProcessor.from_pretrained(
    TELUGU_MODEL
)

te_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    TELUGU_MODEL,
    torch_dtype=DTYPE
)

te_model.to(DEVICE)
te_model.eval()

print("Telugu model loaded.")


# ============================================================
# 3. LOAD MULTILINGUAL WHISPER SMALL
# ============================================================

print("\nLoading Whisper Small...")

multi_processor = AutoProcessor.from_pretrained(
    MULTILINGUAL_MODEL
)

multi_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MULTILINGUAL_MODEL,
    torch_dtype=DTYPE
)

multi_model.to(DEVICE)
multi_model.eval()

print("Whisper Small loaded.")


# ============================================================
# 4. LOAD AUDIO
# ============================================================

def load_audio(audio_path):

    audio, sample_rate = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    return audio, sample_rate


# ============================================================
# 5. LANGUAGE DETECTION
# ============================================================

def detect_language(audio_path):

    audio, sample_rate = load_audio(
        audio_path
    )

    inputs = multi_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    decoder_start_token_id = (
        multi_model.config.decoder_start_token_id
    )

    decoder_input_ids = torch.tensor(
        [[decoder_start_token_id]],
        device=DEVICE
    )

    with torch.no_grad():

        outputs = multi_model(
            input_features=input_features,
            decoder_input_ids=decoder_input_ids
        )

        logits = outputs.logits[:, -1, :]

        probabilities = torch.softmax(
            logits,
            dim=-1
        )[0]

    tokenizer = multi_processor.tokenizer
    vocab = tokenizer.get_vocab()

    languages = {
        "te": "Telugu",
        "hi": "Hindi",
        "en": "English"
    }

    scores = {}

    for code in languages:

        token = f"<|{code}|>"

        token_id = vocab.get(token)

        if token_id is not None:

            scores[code] = probabilities[
                token_id
            ].item()

    print("\nLanguage probabilities:")

    for code, name in languages.items():

        print(
            f"{name:10s}: "
            f"{scores.get(code, 0.0):.2%}"
        )

    if not scores:

        return {
            "code": "en",
            "language": "English",
            "confidence": 0.0,
            "scores": {}
        }

    ranked = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    best_code = ranked[0][0]
    best_score = ranked[0][1]

    second_score = (
        ranked[1][1]
        if len(ranked) > 1
        else 0.0
    )

    # --------------------------------------------------------
    # Telugu detection
    # --------------------------------------------------------

    if best_code == "te":

        if (
            best_score >= 0.20
            and best_score >= second_score * 1.20
        ):

            final_code = "te"

        else:

            alternatives = [
                item
                for item in ranked
                if item[0] != "te"
            ]

            if alternatives:

                final_code = alternatives[0][0]

            else:

                final_code = best_code

    else:

        final_code = best_code

    return {
        "code": final_code,
        "language": languages[final_code],
        "confidence": scores.get(
            final_code,
            0.0
        ),
        "scores": scores
    }


# ============================================================
# 6. CHECK WHETHER TRANSCRIPTION LOOKS ENGLISH
# ============================================================

def english_text_ratio(text):

    """
    Calculate how much of the transcription consists
    of English/Latin alphabet characters.

    Example:

        Hello how are you
        -> high ratio

        مرحباً
        -> 0

        తెలుగు
        -> 0
    """

    if not text:
        return 0.0

    letters = [
        ch
        for ch in text
        if ch.isalpha()
    ]

    if not letters:
        return 0.0

    english_letters = [
        ch
        for ch in letters
        if (
            ("a" <= ch.lower() <= "z")
        )
    ]

    return (
        len(english_letters)
        / len(letters)
    )


def looks_like_english(text):

    """
    Decide whether the generated transcription
    actually looks like English.

    We require at least some Latin characters.
    """

    if not text:
        return False

    ratio = english_text_ratio(text)

    print(
        f"English character ratio: {ratio:.2%}"
    )

    # --------------------------------------------------------
    # English text normally contains a significant amount
    # of Latin characters.
    #
    # Example:
    #
    # Hello how are you
    # -> ~100%
    #
    # مرحباً
    # -> 0%
    #
    # తెలుగు
    # -> 0%
    # --------------------------------------------------------

    return ratio >= 0.40


# ============================================================
# 7. TELUGU TRANSCRIPTION
# ============================================================

def transcribe_telugu(audio_path):

    print(
        "\nUsing Telugu Whisper model..."
    )

    audio, sample_rate = load_audio(
        audio_path
    )

    inputs = te_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    # --------------------------------------------------------
    # Force Telugu
    # --------------------------------------------------------

    forced_decoder_ids = (
        te_processor.tokenizer.get_decoder_prompt_ids(
            language="te",
            task="transcribe"
        )
    )

    te_model.config.forced_decoder_ids = (
        forced_decoder_ids
    )

    with torch.no_grad():

        predicted_ids = te_model.generate(
            input_features
        )

    transcript = te_processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    print(
        "\nTelugu model output:"
    )
    print(transcript)

    return transcript


# ============================================================
# 8. HINDI TRANSCRIPTION
# ============================================================

def transcribe_hindi(audio_path):

    print(
        "\nUsing Whisper Small for Hindi..."
    )

    audio, sample_rate = load_audio(
        audio_path
    )

    inputs = multi_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    forced_decoder_ids = (
        multi_processor.tokenizer.get_decoder_prompt_ids(
            language="hi",
            task="transcribe"
        )
    )

    multi_model.config.forced_decoder_ids = (
        forced_decoder_ids
    )

    with torch.no_grad():

        predicted_ids = multi_model.generate(
            input_features
        )

    transcript = multi_processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    return transcript


# ============================================================
# 9. ENGLISH TRANSCRIPTION
# ============================================================

def transcribe_english(audio_path):

    print(
        "\nUsing Whisper Small for English..."
    )

    audio, sample_rate = load_audio(
        audio_path
    )

    inputs = multi_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    forced_decoder_ids = (
        multi_processor.tokenizer.get_decoder_prompt_ids(
            language="en",
            task="transcribe"
        )
    )

    multi_model.config.forced_decoder_ids = (
        forced_decoder_ids
    )

    with torch.no_grad():

        predicted_ids = multi_model.generate(
            input_features
        )

    transcript = multi_processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    print(
        "\nEnglish model output:"
    )
    print(transcript)

    return transcript


# ============================================================
# 10. MAIN SPEECH-TO-TEXT
# ============================================================

def speech_to_text(audio_path):

    if not audio_path:

        return (
            "No audio recorded.",
            ""
        )

    print("\n" + "=" * 60)
    print("PROCESSING AUDIO")
    print("=" * 60)

    # --------------------------------------------------------
    # STEP 1
    # Detect language
    # --------------------------------------------------------

    detection = detect_language(
        audio_path
    )

    code = detection["code"]
    language = detection["language"]
    confidence = detection["confidence"]
    scores = detection["scores"]

    # --------------------------------------------------------
    # STEP 2
    # Select transcription
    # --------------------------------------------------------

    if code == "te":

        print(
            "\nDetected Telugu."
        )

        transcript = transcribe_telugu(
            audio_path
        )

        final_language = "Telugu"

    elif code == "hi":

        print(
            "\nDetected Hindi."
        )

        transcript = transcribe_hindi(
            audio_path
        )

        final_language = "Hindi"

    else:

        # ----------------------------------------------------
        # English detected
        # ----------------------------------------------------

        print(
            "\nDetected English."
        )

        transcript = transcribe_english(
            audio_path
        )

        # ----------------------------------------------------
        # IMPORTANT FALLBACK
        #
        # If English was detected but the generated text
        # doesn't actually look English, use Telugu model.
        # ----------------------------------------------------

        if not looks_like_english(
            transcript
        ):

            print("\n" + "=" * 60)
            print(
                "ENGLISH TRANSCRIPTION VALIDATION FAILED"
            )
            print("=" * 60)

            print(
                "English transcription was:"
            )
            print(transcript)

            print(
                "\nFalling back to Telugu model..."
            )

            transcript = transcribe_telugu(
                audio_path
            )

            final_language = "Telugu"

        else:

            final_language = "English"

    # --------------------------------------------------------
    # STEP 3
    # Language information
    # --------------------------------------------------------

    language_info = (
        f"Detected Language: {language}\n\n"
        f"Confidence: {confidence:.2%}\n\n"
        f"Language Probabilities:\n"
        f"Telugu:   {scores.get('te', 0.0):.2%}\n"
        f"Hindi:    {scores.get('hi', 0.0):.2%}\n"
        f"English:  {scores.get('en', 0.0):.2%}\n\n"
        f"Transcription Model: {final_language}"
    )

    print("\n" + "=" * 60)
    print("FINAL RESULT")
    print("=" * 60)

    print(language_info)

    print("\nFinal transcription:")
    print(transcript)

    return (
        language_info,
        transcript
    )


# ============================================================
# 11. GRADIO UI
# ============================================================

with gr.Blocks(
    title="Multilingual Speech to Text"
) as demo:

    gr.Markdown(
        """
        # Multilingual Speech-to-Text

        Record speech in:

        - Telugu
        - Hindi
        - English
        - Telugu + English

        The system first detects the language and then
        validates the generated transcription.
        """
    )

    # --------------------------------------------------------
    # AUDIO
    # --------------------------------------------------------

    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        format="wav",
        label="Record Audio"
    )

    # --------------------------------------------------------
    # BUTTON
    # --------------------------------------------------------

    transcribe_button = gr.Button(
        "Transcribe",
        variant="primary"
    )

    # --------------------------------------------------------
    # LANGUAGE
    # --------------------------------------------------------

    language_output = gr.Textbox(
        label="Language Detection",
        lines=9,
        interactive=False
    )

    # --------------------------------------------------------
    # TRANSCRIPTION
    # --------------------------------------------------------

    transcript_output = gr.Textbox(
        label="Transcription",
        lines=8,
        interactive=False
    )

    # --------------------------------------------------------
    # EVENT
    # --------------------------------------------------------

    transcribe_button.click(
        fn=speech_to_text,
        inputs=[
            audio_input
        ],
        outputs=[
            language_output,
            transcript_output
        ]
    )


# ============================================================
# 12. LAUNCH
# ============================================================

print("\nStarting Gradio...")

demo.launch()

In [ ]:
# ============================================================
# WHISPER SPEECH-TO-ENGLISH TRANSLATION
#
# Telugu / Hindi / other supported languages
#             ↓
#        Whisper Small
#             ↓
#        English Text
# ============================================================

import torch
import librosa
import gradio as gr

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "openai/whisper-medium"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)

print("=" * 60)
print("WHISPER SPEECH-TO-ENGLISH TRANSLATION")
print("=" * 60)

print("Model:", MODEL_NAME)
print("Device:", DEVICE)


# ============================================================
# 2. LOAD WHISPER MODEL
# ============================================================

print("\nLoading Whisper Small...")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE
)

model.to(DEVICE)
model.eval()

print("Whisper Small loaded.")


# ============================================================
# 3. LOAD AUDIO
# ============================================================

def load_audio(audio_path):

    audio, sample_rate = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    return audio, sample_rate


# ============================================================
# 4. SPEECH TO ENGLISH TRANSLATION
# ============================================================

def speech_to_english(audio_path):

    if not audio_path:

        return "Please record an audio file."

    print("\n" + "=" * 60)
    print("PROCESSING AUDIO")
    print("=" * 60)

    print("Audio:", audio_path)

    # --------------------------------------------------------
    # Load audio
    # --------------------------------------------------------

    audio, sample_rate = load_audio(
        audio_path
    )

    # --------------------------------------------------------
    # Prepare Whisper input
    # --------------------------------------------------------

    inputs = processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # task="translate"
    #
    # This tells Whisper:
    #
    # Speech in any supported language
    #              ↓
    #          English text
    # --------------------------------------------------------

    forced_decoder_ids = (
        processor.tokenizer.get_decoder_prompt_ids(
            task="translate"
        )
    )

    model.config.forced_decoder_ids = (
        forced_decoder_ids
    )

    # --------------------------------------------------------
    # Generate English translation
    # --------------------------------------------------------

    with torch.no_grad():

        predicted_ids = model.generate(
            input_features
        )

    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    translated_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    print("\nEnglish Translation:")
    print(translated_text)

    return translated_text


# ============================================================
# 5. GRADIO UI
# ============================================================

with gr.Blocks(
    title="Whisper Speech to English"
) as demo:

    gr.Markdown(
        """
        # Whisper Speech-to-English Translation

        Record speech in **Telugu, Hindi, or another supported
        language**.

        Whisper will translate the spoken audio directly into
        **English text**.
        """
    )

    # --------------------------------------------------------
    # AUDIO INPUT
    # --------------------------------------------------------

    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        format="wav",
        label="Record Speech"
    )

    # --------------------------------------------------------
    # BUTTON
    # --------------------------------------------------------

    translate_button = gr.Button(
        "Translate to English",
        variant="primary"
    )

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    english_output = gr.Textbox(
        label="English Translation",
        lines=8,
        interactive=False
    )

    # --------------------------------------------------------
    # EVENT
    # --------------------------------------------------------

    translate_button.click(
        fn=speech_to_english,
        inputs=[
            audio_input
        ],
        outputs=[
            english_output
        ]
    )


# ============================================================
# 6. LAUNCH
# ============================================================

print("\nStarting Gradio...")

demo.launch()

# chatbot

In [ ]:
a

In [ ]:
%pip install google-cloud-speech

In [ ]:
# ============================================================
# AI INTERVIEW CHATBOT
# Multilingual Voice:
#   Telugu -> vasista22/whisper-telugu-small
#   Hindi  -> openai/whisper-small
#   English -> openai/whisper-small
#
# Audio:
#   Gradio temp file
#       -> /uploads/voice_<uuid>.wav
#       -> language detection
#       -> transcription
#       -> interview answer
#       -> evaluation
#       -> delete /uploads file
#
# Evaluation is saved to JSON but NEVER shown below the answer.
# ============================================================

import os
import json
import uuid
import shutil
from pathlib import Path

import torch
import gradio as gr
import librosa

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe"
)

OUTPUT_DIR = PROJECT_ROOT / "interview_results"
UPLOAD_DIR = PROJECT_ROOT / "uploads"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. WHISPER CONFIGURATION
# ============================================================

TELUGU_MODEL_NAME = "vasista22/whisper-telugu-small"
MULTILINGUAL_MODEL_NAME = "openai/whisper-small"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    DTYPE = torch.float16
else:
    DTYPE = torch.float32

print("=" * 60)
print("VOICE MODEL CONFIGURATION")
print("=" * 60)
print("Device:", DEVICE)
print("Telugu model:", TELUGU_MODEL_NAME)
print("Hindi/English model:", MULTILINGUAL_MODEL_NAME)


# ============================================================
# 3. LOAD TELUGU WHISPER MODEL
# ============================================================

print("\nLoading Telugu Whisper model...")

telugu_processor = AutoProcessor.from_pretrained(
    TELUGU_MODEL_NAME
)

telugu_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    TELUGU_MODEL_NAME,
    torch_dtype=DTYPE
)

telugu_model.to(DEVICE)
telugu_model.eval()

print("Telugu Whisper model loaded.")


# ============================================================
# 4. LOAD MULTILINGUAL WHISPER MODEL
#    Used for:
#      - language detection
#      - Hindi transcription
#      - English transcription
# ============================================================

print("\nLoading multilingual Whisper Small model...")

multi_processor = AutoProcessor.from_pretrained(
    MULTILINGUAL_MODEL_NAME
)

multi_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MULTILINGUAL_MODEL_NAME,
    torch_dtype=DTYPE
)

multi_model.to(DEVICE)
multi_model.eval()

print("Multilingual Whisper model loaded.")


# ============================================================
# 5. SAVE GRADIO AUDIO INTO /uploads
# ============================================================

def save_audio_to_uploads(source_audio_path):
    """
    Copy Gradio's temporary audio file into:

        PROJECT_ROOT/uploads/

    Returns:
        Path of copied audio file.
    """

    if not source_audio_path:
        raise ValueError("No audio file was provided.")

    source_path = Path(str(source_audio_path))

    if not source_path.exists():
        raise FileNotFoundError(
            f"Audio file does not exist: {source_path}"
        )

    audio_id = uuid.uuid4().hex

    destination = UPLOAD_DIR / f"voice_{audio_id}.wav"

    shutil.copy2(
        source_path,
        destination
    )

    print("\nAudio copied to:")
    print(destination)

    return destination


# ============================================================
# 6. LOAD AUDIO
# ============================================================

def load_audio_16k(audio_path):
    """
    Load audio as:
      - mono
      - 16 kHz
      - float32
    """

    audio, sample_rate = librosa.load(
        str(audio_path),
        sr=16000,
        mono=True
    )

    return audio, sample_rate


# ============================================================
# 7. LANGUAGE DETECTION USING WHISPER SMALL
# ============================================================

def detect_language(audio_path):
    """
    Detect whether speech is:
        Telugu -> te
        Hindi  -> hi
        English -> en

    Uses openai/whisper-small.
    """

    print("\nDetecting language...")

    audio, sample_rate = load_audio_16k(audio_path)

    inputs = multi_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    # --------------------------------------------------------
    # Get decoder start token
    # --------------------------------------------------------

    decoder_start_token_id = (
        multi_model.config.decoder_start_token_id
    )

    decoder_input_ids = torch.tensor(
        [[decoder_start_token_id]],
        device=DEVICE
    )

    # --------------------------------------------------------
    # First decoder step gives language probabilities
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = multi_model(
            input_features=input_features,
            decoder_input_ids=decoder_input_ids
        )

        logits = outputs.logits[:, -1, :]

        probabilities = torch.softmax(
            logits,
            dim=-1
        )

    # --------------------------------------------------------
    # Whisper language tokens
    # --------------------------------------------------------

    tokenizer = multi_processor.tokenizer

    language_candidates = {
        "te": "Telugu",
        "hi": "Hindi",
        "en": "English",
    }

    language_scores = {}

    vocab = tokenizer.get_vocab()

    for language_code in language_candidates:

        token = f"<|{language_code}|>"

        token_id = vocab.get(token)

        if token_id is not None:

            score = probabilities[
                0,
                token_id
            ].item()

            language_scores[
                language_code
            ] = score

    if not language_scores:

        print(
            "Could not determine language tokens."
        )

        return {
            "language": "en",
            "language_name": "English",
            "confidence": 0.0
        }

    detected_language = max(
        language_scores,
        key=language_scores.get
    )

    confidence = language_scores[
        detected_language
    ]

    language_name = language_candidates[
        detected_language
    ]

    print(
        f"Detected language: "
        f"{language_name} "
        f"({confidence:.4f})"
    )

    print(
        "Language scores:",
        language_scores
    )

    return {
        "language": detected_language,
        "language_name": language_name,
        "confidence": confidence
    }


# ============================================================
# 8. TELUGU TRANSCRIPTION
# ============================================================

def transcribe_telugu(audio_path):
    """
    Transcribe Telugu using:
        vasista22/whisper-telugu-small
    """

    print("\nTranscribing Telugu audio...")

    audio, sample_rate = load_audio_16k(
        audio_path
    )

    inputs = telugu_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    # --------------------------------------------------------
    # Important:
    # Set forced_decoder_ids on model.config.
    #
    # Do NOT pass forced_decoder_ids directly to generate().
    # --------------------------------------------------------

    forced_decoder_ids = (
        telugu_processor.tokenizer.get_decoder_prompt_ids(
            language="te",
            task="transcribe"
        )
    )

    telugu_model.config.forced_decoder_ids = (
        forced_decoder_ids
    )

    with torch.no_grad():

        predicted_ids = telugu_model.generate(
            input_features
        )

    transcription = (
        telugu_processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]
        .strip()
    )

    print("Telugu transcription:")
    print(transcription)

    return transcription


# ============================================================
# 9. HINDI TRANSCRIPTION
# ============================================================

def transcribe_hindi(audio_path):
    """
    Transcribe Hindi using:
        openai/whisper-small
    """

    print("\nTranscribing Hindi audio...")

    audio, sample_rate = load_audio_16k(
        audio_path
    )

    inputs = multi_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    forced_decoder_ids = (
        multi_processor.tokenizer.get_decoder_prompt_ids(
            language="hi",
            task="transcribe"
        )
    )

    multi_model.config.forced_decoder_ids = (
        forced_decoder_ids
    )

    with torch.no_grad():

        predicted_ids = multi_model.generate(
            input_features
        )

    transcription = (
        multi_processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]
        .strip()
    )

    print("Hindi transcription:")
    print(transcription)

    return transcription


# ============================================================
# 10. ENGLISH TRANSCRIPTION
# ============================================================

def transcribe_english(audio_path):
    """
    Transcribe English using:
        openai/whisper-small
    """

    print("\nTranscribing English audio...")

    audio, sample_rate = load_audio_16k(
        audio_path
    )

    inputs = multi_processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    forced_decoder_ids = (
        multi_processor.tokenizer.get_decoder_prompt_ids(
            language="en",
            task="transcribe"
        )
    )

    multi_model.config.forced_decoder_ids = (
        forced_decoder_ids
    )

    with torch.no_grad():

        predicted_ids = multi_model.generate(
            input_features
        )

    transcription = (
        multi_processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]
        .strip()
    )

    print("English transcription:")
    print(transcription)

    return transcription


# ============================================================
# 11. MULTILINGUAL TRANSCRIPTION
# ============================================================

def transcribe_multilingual_audio(audio_path):
    """
    Complete voice pipeline:

        Gradio audio
              |
              v
        /uploads
              |
              v
       Language detection
              |
       +------+-------+
       |      |       |
       v      v       v
      TE     HI      EN
       |      |       |
       v      v       v
    Telugu   Whisper  Whisper
    Whisper   Hindi   English
       |      |       |
       +------+-------+
              |
              v
        Transcription
              |
              v
       Delete audio
    """

    saved_audio_path = None

    try:

        # ----------------------------------------------------
        # Save audio to /uploads
        # ----------------------------------------------------

        saved_audio_path = save_audio_to_uploads(
            audio_path
        )

        # ----------------------------------------------------
        # Detect language
        # ----------------------------------------------------

        language_info = detect_language(
            saved_audio_path
        )

        language = language_info["language"]

        # ----------------------------------------------------
        # Route to correct model
        # ----------------------------------------------------

        if language == "te":

            transcript = transcribe_telugu(
                saved_audio_path
            )

        elif language == "hi":

            transcript = transcribe_hindi(
                saved_audio_path
            )

        else:

            # English
            transcript = transcribe_english(
                saved_audio_path
            )

        # ----------------------------------------------------
        # Return transcription information
        # ----------------------------------------------------

        return {
            "language": language_info["language"],
            "language_name": language_info["language_name"],
            "confidence": language_info["confidence"],
            "transcript": transcript
        }

    finally:

        # ----------------------------------------------------
        # ALWAYS DELETE AUDIO
        # ----------------------------------------------------

        if (
            saved_audio_path is not None
            and saved_audio_path.exists()
        ):

            try:

                saved_audio_path.unlink()

                print(
                    "\nDeleted uploaded audio:"
                )
                print(saved_audio_path)

            except Exception as delete_error:

                print(
                    "Could not delete audio:",
                    delete_error
                )


# ============================================================
# 12. APPLICATION STATE
# ============================================================

app_state = {
    "interview": None,
    "resume_analysis": None,
    "resume_text": "",
    "job_description": "",
    "started": False
}


# ============================================================
# 13. PROCESS RESUME + JOB DESCRIPTION
# ============================================================

def process_documents(
    resume_file,
    job_description_file
):

    try:

        if not resume_file:

            return (
                "Please upload a resume.",
                "",
                gr.update(interactive=False)
            )

        if not job_description_file:

            return (
                "Please upload a job description.",
                "",
                gr.update(interactive=False)
            )

        print("\nExtracting resume...")

        resume_text = extract_document_text(
            resume_file
        )

        if not resume_text.strip():

            return (
                "Resume is empty or could not be extracted.",
                "",
                gr.update(interactive=False)
            )

        print("Extracting job description...")

        job_description = extract_document_text(
            job_description_file
        )

        if not job_description.strip():

            return (
                "Job description is empty or could not be extracted.",
                "",
                gr.update(interactive=False)
            )

        print("Analyzing resume...")

        resume_analysis = analyze_resume(
            resume_text
        )

        # ----------------------------------------------------
        # Save state
        # ----------------------------------------------------

        app_state["resume_text"] = resume_text
        app_state["job_description"] = job_description
        app_state["resume_analysis"] = resume_analysis
        app_state["interview"] = None
        app_state["started"] = False

        # ----------------------------------------------------
        # Display analysis
        # ----------------------------------------------------

        if hasattr(
            resume_analysis,
            "model_dump"
        ):

            analysis_dict = (
                resume_analysis.model_dump()
            )

        else:

            analysis_dict = (
                resume_analysis.dict()
            )

        analysis_text = json.dumps(
            analysis_dict,
            indent=2,
            ensure_ascii=False
        )

        return (
            "Resume and job description processed successfully.",
            analysis_text,
            gr.update(interactive=True)
        )

    except Exception as e:

        print(
            "\nDocument processing error:",
            repr(e)
        )

        return (
            f"Error: {str(e)}",
            "",
            gr.update(interactive=False)
        )


# ============================================================
# 14. START INTERVIEW
# ============================================================

def start_interview():

    try:

        if app_state["resume_analysis"] is None:

            return (
                [],
                "Please process the resume first.",
                gr.update(interactive=False),
                gr.update(interactive=False),
                gr.update(interactive=False)
            )

        # ----------------------------------------------------
        # Create interview agent
        # ----------------------------------------------------

        interview = create_interview_agent(
            llm=llm,
            resume_analysis=app_state["resume_analysis"],
            evaluate_answer=evaluate_answer,
            job_description=app_state["job_description"],
            max_questions=10,
            output_dir=OUTPUT_DIR
        )

        app_state["interview"] = interview

        # ----------------------------------------------------
        # Generate first question
        # ----------------------------------------------------

        question = interview.start()

        app_state["started"] = True

        print("\nINTERVIEW STARTED")
        print("Question:", question)

        return (
            interview.messages,
            "Interview started.",
            gr.update(interactive=True),
            gr.update(interactive=True),
            gr.update(interactive=True)
        )

    except Exception as e:

        print(
            "\nInterview start error:",
            repr(e)
        )

        return (
            [],
            f"Error: {str(e)}",
            gr.update(interactive=False),
            gr.update(interactive=False),
            gr.update(interactive=False)
        )


# ============================================================
# 15. SUBMIT TEXT ANSWER
# ============================================================

def submit_text_answer(answer):

    try:

        if not app_state["started"]:

            return (
                [],
                "Please start the interview first.",
                ""
            )

        if not answer or not answer.strip():

            return (
                app_state["interview"].messages,
                "Please enter an answer.",
                ""
            )

        interview = app_state["interview"]

        # ----------------------------------------------------
        # Submit answer
        #
        # Evaluation happens internally.
        # It is NOT returned to Gradio.
        # ----------------------------------------------------

        interview.submit_answer(
            answer.strip()
        )

        if interview.finished:

            status = (
                "Interview completed. "
                "Evaluation saved to JSON."
            )

        else:

            status = (
                "Answer submitted. "
                "Next question generated."
            )

        return (
            interview.messages,
            status,
            ""
        )

    except Exception as e:

        print(
            "\nText answer error:",
            repr(e)
        )

        return (
            app_state["interview"].messages
            if app_state["interview"]
            else [],
            f"Error: {str(e)}",
            ""
        )


# ============================================================
# 16. SUBMIT VOICE ANSWER
# ============================================================

def submit_voice_answer(audio_path):

    print("\n" + "=" * 60)
    print("VOICE SUBMIT BUTTON CLICKED")
    print("=" * 60)

    try:

        if not app_state["started"]:

            return (
                [],
                "Please start the interview first.",
                "Please start the interview first."
            )

        if not audio_path:

            return (
                app_state["interview"].messages,
                "",
                "Please record your answer first."
            )

        print("Audio path:", audio_path)

        # ----------------------------------------------------
        # Multilingual transcription
        # ----------------------------------------------------

        result = transcribe_multilingual_audio(
            audio_path
        )

        transcript = result["transcript"]
        language_name = result["language_name"]
        confidence = result["confidence"]

        print("\nLanguage:", language_name)
        print("Confidence:", confidence)
        print("Transcript:", transcript)

        if not transcript.strip():

            return (
                app_state["interview"].messages,
                "",
                "Could not understand the audio."
            )

        # ----------------------------------------------------
        # Submit transcript to interview
        #
        # Evaluation remains hidden.
        # ----------------------------------------------------

        interview = app_state["interview"]

        interview.submit_answer(
            transcript
        )

        if interview.finished:

            status = (
                f"Voice answer submitted "
                f"({language_name}). "
                f"Interview completed."
            )

        else:

            status = (
                f"Voice answer submitted "
                f"({language_name}). "
                f"Next question generated."
            )

        # ----------------------------------------------------
        # Show ONLY transcription to user.
        # Do NOT show evaluation.
        # ----------------------------------------------------

        transcription_display = (
            f"[Detected language: {language_name}]\n\n"
            f"{transcript}"
        )

        return (
            interview.messages,
            transcription_display,
            status
        )

    except Exception as e:

        print(
            "\nVoice answer error:",
            repr(e)
        )

        return (
            app_state["interview"].messages
            if app_state["interview"]
            else [],
            "",
            f"Voice processing error: {str(e)}"
        )


# ============================================================
# 17. VIEW SAVED JSON RESULTS
# ============================================================

def view_results():

    try:

        interview = app_state["interview"]

        if interview is None:

            return "No interview has been started."

        results = interview.get_results()

        return json.dumps(
            results,
            indent=2,
            ensure_ascii=False
        )

    except Exception as e:

        return f"Error loading results: {str(e)}"


# ============================================================
# 18. RESET APPLICATION
# ============================================================

def reset_application():

    # --------------------------------------------------------
    # Delete any leftover voice files
    # --------------------------------------------------------

    for audio_file in UPLOAD_DIR.glob(
        "voice_*"
    ):

        try:

            audio_file.unlink()

        except Exception:
            pass

    # --------------------------------------------------------
    # Reset application state
    # --------------------------------------------------------

    app_state["interview"] = None
    app_state["resume_analysis"] = None
    app_state["resume_text"] = ""
    app_state["job_description"] = ""
    app_state["started"] = False

    return (
        None,       # resume file
        None,       # JD file
        "",         # document status
        "",         # resume analysis
        [],         # chatbot
        "",         # interview status
        "",         # transcription
        "",         # results
        gr.update(interactive=False),  # start
        gr.update(interactive=False),  # text input
        gr.update(interactive=False),  # text button
        gr.update(interactive=False),  # voice button
    )


# ============================================================
# 19. GRADIO UI
# ============================================================

with gr.Blocks(
    title="AI Interview Chatbot"
) as demo:

    gr.Markdown(
        """
        # AI Interview Chatbot

        Upload your **resume** and **job description** to start an
        adaptive AI interview.

        You can answer using:

        - Text
        - Telugu voice
        - Hindi voice
        - English voice

        Voice language is detected automatically.
        """
    )

    # --------------------------------------------------------
    # DOCUMENT UPLOAD
    # --------------------------------------------------------

    with gr.Row():

        resume_file = gr.File(
            label="Resume",
            file_types=[
                ".pdf",
                ".docx",
                ".txt"
            ],
            type="filepath"
        )

        job_description_file = gr.File(
            label="Job Description",
            file_types=[
                ".pdf",
                ".docx",
                ".txt"
            ],
            type="filepath"
        )

    process_button = gr.Button(
        "Process Documents",
        variant="primary"
    )

    document_status = gr.Textbox(
        label="Document Status",
        interactive=False
    )

    resume_analysis_box = gr.Textbox(
        label="Resume Analysis",
        lines=15,
        interactive=False
    )

    # --------------------------------------------------------
    # START INTERVIEW
    # --------------------------------------------------------

    start_button = gr.Button(
        "Start Interview",
        variant="primary",
        interactive=False
    )

    # --------------------------------------------------------
    # CHAT
    # --------------------------------------------------------

    chatbot = gr.Chatbot(
        label="Interview",
        height=500
    )

    interview_status = gr.Textbox(
        label="Interview Status",
        interactive=False
    )

    # --------------------------------------------------------
    # TEXT ANSWER
    # --------------------------------------------------------

    text_answer = gr.Textbox(
        label="Type Your Answer",
        placeholder="Type your answer here...",
        lines=4,
        interactive=False
    )

    send_text_button = gr.Button(
        "Submit Text Answer",
        interactive=False
    )

    # --------------------------------------------------------
    # VOICE ANSWER
    # --------------------------------------------------------

    gr.Markdown(
        "### Voice Answer"
    )

    voice_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        format="wav",
        label="Record your answer"
    )

    send_voice_button = gr.Button(
        "Submit Voice Answer",
        variant="primary",
        interactive=False
    )

    transcription_box = gr.Textbox(
        label="Voice Transcription",
        lines=4,
        interactive=False
    )

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    with gr.Row():

        results_button = gr.Button(
            "View Interview Results"
        )

        reset_button = gr.Button(
            "Reset"
        )

    results_box = gr.Textbox(
        label="Interview Results JSON",
        lines=20,
        interactive=False
    )

    # ========================================================
    # GRADIO EVENTS
    # ========================================================

    # --------------------------------------------------------
    # Process documents
    # --------------------------------------------------------

    process_button.click(
        fn=process_documents,
        inputs=[
            resume_file,
            job_description_file
        ],
        outputs=[
            document_status,
            resume_analysis_box,
            start_button
        ]
    )

    # --------------------------------------------------------
    # Start interview
    # --------------------------------------------------------

    start_button.click(
        fn=start_interview,
        inputs=[],
        outputs=[
            chatbot,
            interview_status,
            text_answer,
            send_text_button,
            send_voice_button
        ]
    )

    # --------------------------------------------------------
    # Submit text
    # --------------------------------------------------------

    send_text_button.click(
        fn=submit_text_answer,
        inputs=[
            text_answer
        ],
        outputs=[
            chatbot,
            interview_status,
            text_answer
        ]
    )

    # Press Enter / submit textbox
    text_answer.submit(
        fn=submit_text_answer,
        inputs=[
            text_answer
        ],
        outputs=[
            chatbot,
            interview_status,
            text_answer
        ]
    )

    # --------------------------------------------------------
    # Submit voice
    # --------------------------------------------------------

    send_voice_button.click(
        fn=submit_voice_answer,
        inputs=[
            voice_input
        ],
        outputs=[
            chatbot,
            transcription_box,
            interview_status
        ]
    )

    # --------------------------------------------------------
    # View results
    # --------------------------------------------------------

    results_button.click(
        fn=view_results,
        inputs=[],
        outputs=[
            results_box
        ]
    )

    # --------------------------------------------------------
    # Reset
    # --------------------------------------------------------

    reset_button.click(
        fn=reset_application,
        inputs=[],
        outputs=[
            resume_file,
            job_description_file,
            document_status,
            resume_analysis_box,
            chatbot,
            interview_status,
            transcription_box,
            results_box,
            start_button,
            text_answer,
            send_text_button,
            send_voice_button
        ]
    )


# ============================================================
# 20. LAUNCH
# ============================================================

print("\n" + "=" * 60)
print("Launching AI Interview Chatbot...")
print("=" * 60)

demo.launch()

In [ ]:
%pip install mysql-connector-python pandas openpyxl

# SQL to links

In [34]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [35]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [68]:
db_config = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "Mysql@2025",
    "database": "datapro_lms",
    "connection_timeout": 10
}

In [76]:

def get_lesson_summaries(module_ids, lesson_ids, course_ids, db_config):
    import re
    import requests
    import pandas as pd

    from pathlib import Path
    from sqlalchemy import create_engine, URL
    from dotenv import load_dotenv
    from aipipe import LLM

    # Validate inputs
    if not (len(module_ids) == len(lesson_ids) == len(course_ids)):
        raise ValueError("All ID lists must have equal lengths.")

    if not module_ids:
        return []

    # Configuration
    project_root = Path(
        r"C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe"
    )
    load_dotenv(project_root / ".env")

    llm = LLM.from_config(
        str(project_root / "config" / "llm.yaml")
    )

    # FastAPI document extraction endpoint
    extraction_api_url = "http://127.0.0.1:8000/api/file/extract-text"

    engine = create_engine(
        URL.create(
            drivername="mysql+pymysql",
            username=db_config["user"],
            password=db_config["password"],
            host=db_config["host"],
            port=int(db_config["port"]),
            database=db_config["database"]
        ),
        pool_pre_ping=True
    )

    def extract_document_text(url, timeout=120):
        """Download a Google Doc/Slide and extract its text using FastAPI."""

        match = re.search(
            r"/(document|presentation)/d/([^/?]+)",
            url
        )

        if not match:
            raise ValueError(f"Unsupported Google URL: {url}")

        doc_type, doc_id = match.groups()

        if doc_type == "document":
            export_url = (
                f"https://docs.google.com/document/d/"
                f"{doc_id}/export?format=docx"
            )
            filename = f"{doc_id}.docx"
            mime_type = (
                "application/vnd.openxmlformats-officedocument."
                "wordprocessingml.document"
            )
        else:
            export_url = (
                f"https://docs.google.com/presentation/d/"
                f"{doc_id}/export/pptx"
            )
            filename = f"{doc_id}.pptx"
            mime_type = (
                "application/vnd.openxmlformats-officedocument."
                "presentationml.presentation"
            )

        # Download the exported file.
        download_response = requests.get(
            export_url,
            timeout=timeout,
            allow_redirects=True
        )
        download_response.raise_for_status()

        if "text/html" in download_response.headers.get(
            "Content-Type", ""
        ).lower():
            raise ValueError(
                "Google returned HTML instead of a document. "
                "Check the document's sharing permissions."
            )

        if not download_response.content:
            raise ValueError("Downloaded document is empty.")

        # Send the file directly to the FastAPI extraction endpoint.
        api_response = requests.post(
            extraction_api_url,
            files={
                "file": (
                    filename,
                    download_response.content,
                    mime_type
                )
            },
            timeout=timeout
        )
        api_response.raise_for_status()

        try:
            data = api_response.json()
        except ValueError as exc:
            raise ValueError(
                "Extraction API did not return valid JSON."
            ) from exc

        if not data.get("success", True):
            raise RuntimeError(
                data.get("message", "Extraction API reported failure.")
            )

        text = data.get("text", data.get("extracted_text", ""))

        if not isinstance(text, str) or not text.strip():
            raise ValueError(
                "Extraction API returned empty or invalid text."
            )

        return text.strip()

    try:
        # Read database tables.
        course_content = pd.read_sql(
            "SELECT * FROM tbl_course_content",
            engine
        )
        lesson_contents = pd.read_sql(
            "SELECT * FROM tbl_lesson_contents",
            engine
        )

        # Join course and lesson content.
        merged_df = pd.merge(
            course_content,
            lesson_contents,
            on="lesson_id",
            how="inner",
            suffixes=("_course", "_lesson")
        )

        # Exclude deleted records.
        for col in ["is_deleted_course", "is_deleted_lesson"]:
            if col in merged_df.columns:
                merged_df = merged_df[
                    pd.to_numeric(
                        merged_df[col], errors="coerce"
                    ) == 0
                ]

        id_cols = ["module_id", "lesson_id", "course_id"]

        # Normalize IDs to numeric values.
        for col in id_cols:
            merged_df[col] = pd.to_numeric(
                merged_df[col], errors="coerce"
            )

        requested = pd.DataFrame({
            "module_id": module_ids,
            "lesson_id": lesson_ids,
            "course_id": course_ids
        })

        for col in id_cols:
            requested[col] = pd.to_numeric(
                requested[col], errors="coerce"
            )

        # Match exact module, lesson and course combinations.
        matched = merged_df.merge(
            requested.drop_duplicates(),
            on=id_cols,
            how="inner",
            validate="many_to_many"
        )

        print("Requested combinations:")
        print(requested.to_string(index=False))

        print("\nMatched combinations:")
        if matched.empty:
            print("No matching records found.")
        else:
            print(
                matched[id_cols]
                .drop_duplicates()
                .to_string(index=False)
            )

        results = []

        for (module_id, lesson_id, course_id), group in matched.groupby(
            id_cols, sort=False
        ):
            urls = (
                group["content_url"]
                .dropna()
                .astype(str)
                .str.strip()
            )
            urls = urls[urls.ne("")].drop_duplicates().tolist()

            texts = []
            errors = []

            # Extract text from each document through FastAPI.
            for url in urls:
                try:
                    text = extract_document_text(url)

                    if text:
                        texts.append(text)
                    else:
                        errors.append(
                            f"No text extracted from {url}"
                        )

                except Exception as exc:
                    errors.append(f"{url}: {exc}")

            # No usable source documents.
            if not texts:
                results.append({
                    "course_id": int(course_id),
                    "module_id": int(module_id),
                    "lesson_id": int(lesson_id),
                    "urls": urls,
                    "document_count": 0,
                    "summary": None,
                    "status": "failed",
                    "error": "; ".join(errors) or "No text extracted"
                })
                continue

            # Summarize all successfully extracted documents.
            prompt = f"""
Summarize the following lesson material comprehensively.

Use headings and bullet points. Preserve important definitions,
examples, technical details and formulas. Do not add information
that is not present in the source.

Course ID: {course_id}
Module ID: {module_id}
Lesson ID: {lesson_id}

SOURCE MATERIAL:

{chr(10).join(texts)}
"""

            try:
                response = llm.generate(prompt)

                summary = (
                    response
                    if isinstance(response, str)
                    else response.content
                    if hasattr(response, "content")
                    else str(response)
                )

                if not summary or not summary.strip():
                    raise ValueError(
                        "The LLM returned an empty summary."
                    )

                results.append({
                    "course_id": int(course_id),
                    "module_id": int(module_id),
                    "lesson_id": int(lesson_id),
                    "urls": urls,
                    "document_count": len(texts),
                    "summary": summary.strip(),
                    "status": "success",
                    "errors": errors
                })

            except Exception as exc:
                results.append({
                    "course_id": int(course_id),
                    "module_id": int(module_id),
                    "lesson_id": int(lesson_id),
                    "urls": urls,
                    "document_count": len(texts),
                    "summary": None,
                    "status": "failed",
                    "error": str(exc),
                    "errors": errors
                })

        return results

    finally:
        engine.dispose()

In [78]:
summaries = get_lesson_summaries(
    module_ids=[3, 8],
    lesson_ids=[2, 3],
    course_ids=[1, 1],
    db_config=db_config
)

for item in summaries:
    print(
        item["course_id"],
        item["module_id"],
        item["lesson_id"],
        item["status"]
    )

Requested combinations:
 module_id  lesson_id  course_id
         3          2          1
         8          3          1

Matched combinations:
 module_id  lesson_id  course_id
         3        2.0          1
         8        3.0          1
1 3 2 success
1 8 3 success


In [79]:
summaries

[{'course_id': 1,
  'module_id': 3,
  'lesson_id': 2,
  'urls': ['https://docs.google.com/document/d/1NPvcmIV4lbPonxttQo-fjdqJESV8m58x/edit?usp=sharing&ouid=109528509738013099683&rtpof=true&sd=true',
   'https://docs.google.com/presentation/d/1HQ9uKq9tjM8YtZ5cNmXESZNDoTAPdPaP/edit?usp=sharing&ouid=109528509738013099683&rtpof=true&sd=true'],
  'document_count': 2,
  'summary': '# Course 1, Module 3, Lesson 2.0: Program Onboarding & Digital Literacy\n\n## 1. Program Onboarding\n**Definition:** The process through which a learner becomes familiar with a new educational or training program, encompassing more than just registration.\n\n### Importance of Onboarding\n*   **Clarity:** Establishes program goals, schedules, and expectations.\n*   **Connection:** Introduces learners to instructors, coordinators, peers, and support.\n*   **Access:** Explains how to access materials, assignments, and assessments.\n*   **Confidence:** Builds basic digital skills before technology-dependent activitie

In [80]:
get_lesson_summaries(
    module_ids=[3, 8],
    lesson_ids=[2, 3],
    course_ids=[1, 1],
    db_config=db_config
)

Requested combinations:
 module_id  lesson_id  course_id
         3          2          1
         8          3          1

Matched combinations:
 module_id  lesson_id  course_id
         3        2.0          1
         8        3.0          1


[{'course_id': 1,
  'module_id': 3,
  'lesson_id': 2,
  'urls': ['https://docs.google.com/document/d/1NPvcmIV4lbPonxttQo-fjdqJESV8m58x/edit?usp=sharing&ouid=109528509738013099683&rtpof=true&sd=true',
   'https://docs.google.com/presentation/d/1HQ9uKq9tjM8YtZ5cNmXESZNDoTAPdPaP/edit?usp=sharing&ouid=109528509738013099683&rtpof=true&sd=true'],
  'document_count': 2,
  'summary': '# Course 1, Module 3, Lesson 2.0: Program Onboarding & Digital Literacy\n\n## 1. Program Onboarding\n### 1.1 Definition and Purpose\n**Program Onboarding** is the process of familiarizing a learner with a new educational or training program. It extends beyond registration to ensure learners understand:\n* Program purpose, goals, and learning expectations.\n* Available resources, communication channels, and support systems.\n* Rules, methods of assessment, and schedules.\n\n### 1.2 Importance of Onboarding\nOnboarding creates a common starting point for learners with varying levels of academic and digital experien

# Personolized learning

In [65]:
summaries = get_lesson_summaries(
    module_ids=[8],
    lesson_ids=[11],
    course_ids=[1],
    db_config=db_config
)

for item in summaries:
    print(
        item["course_id"],
        item["module_id"],
        item["lesson_id"],
        item["status"]
    )

Requested combinations:
 module_id  lesson_id  course_id
         8         11          1

Matched combinations:
 module_id  lesson_id  course_id
         8       11.0          1
1 8 11 failed


In [66]:
summaries

[{'course_id': 1,
  'module_id': 8,
  'lesson_id': 11,
  'urls': ['https://docs.google.com/document/d/1jFX8k6CPuzO76KEv2SBHv6GKrh8d0fAQ/edit?usp=sharing&ouid=113135456377714534885&rtpof=true&sd=true',
   'https://docs.google.com/presentation/d/1J1YTqZ5iY2E6g_vDZupy19hghE-bY7Y4/edit?usp=sharing&ouid=113135456377714534885&rtpof=true&sd=true'],
  'summary': None,
  'status': 'failed',
  'error': "https://docs.google.com/document/d/1jFX8k6CPuzO76KEv2SBHv6GKrh8d0fAQ/edit?usp=sharing&ouid=113135456377714534885&rtpof=true&sd=true: name 'extract_document_text' is not defined; https://docs.google.com/presentation/d/1J1YTqZ5iY2E6g_vDZupy19hghE-bY7Y4/edit?usp=sharing&ouid=113135456377714534885&rtpof=true&sd=true: name 'extract_document_text' is not defined"}]

# Personalized Learning

In [81]:

def generate_personalized_lesson_html(
    module_ids,
    course_ids,
    student_id,
    db_config,
    llm,
    output_dir="personalized_lessons",
    max_retries=2
):
    import json
    import re
    import html
    import mysql.connector

    from pathlib import Path
    from datetime import datetime

    # ---------------------------------------------------------
    # 1. Validate inputs
    # ---------------------------------------------------------
    if len(module_ids) != len(course_ids):
        raise ValueError(
            "module_ids and course_ids must have equal lengths."
        )

    if not module_ids:
        return []

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    results = []

    # ---------------------------------------------------------
    # 2. Helper functions
    # ---------------------------------------------------------
    def response_text(response):
        if isinstance(response, str):
            return response

        content = getattr(response, "content", None)

        if isinstance(content, str):
            return content

        if isinstance(content, list):
            return "\n".join(
                item if isinstance(item, str)
                else item.get("text", "")
                if isinstance(item, dict)
                else ""
                for item in content
            )

        return str(response)

    def parse_json(response):
        text = response_text(response).strip()

        text = re.sub(
            r"^\s*```(?:json)?\s*|\s*```\s*$",
            "",
            text,
            flags=re.IGNORECASE
        ).strip()

        decoder = json.JSONDecoder()

        for index, char in enumerate(text):
            if char not in "{[":
                continue

            try:
                value, _ = decoder.raw_decode(text[index:])
                if isinstance(value, dict):
                    return value
            except json.JSONDecodeError:
                continue

        raise ValueError(
            "LLM returned invalid JSON. "
            f"Response ending: {text[-500:]}"
        )

    def invoke_json(prompt):
        last_error = None

        for attempt in range(max_retries + 1):
            current_prompt = prompt

            if attempt:
                current_prompt += """

Your previous response was invalid or incomplete.
Return a shorter, complete JSON object.
Do not use Markdown fences or text outside the JSON.
Ensure all brackets and quotation marks are closed.
"""

            response = llm.invoke(current_prompt)

            try:
                return parse_json(response)
            except (ValueError, json.JSONDecodeError) as exc:
                last_error = exc
                print(
                    f"JSON parsing attempt {attempt + 1} failed: {exc}"
                )

        raise ValueError(
            f"Could not obtain valid JSON: {last_error}"
        )

    def safe_markdown(value):
        if value is None:
            return ""

        value = html.escape(str(value), quote=True)
        output = []
        in_list = False

        for line in value.splitlines():
            line = line.strip()

            if not line:
                if in_list:
                    output.append("</ul>")
                    in_list = False
                continue

            heading = re.match(r"^(#{1,4})\s+(.+)$", line)
            bullet = re.match(r"^[-*]\s+(.+)$", line)

            if heading:
                if in_list:
                    output.append("</ul>")
                    in_list = False

                level = len(heading.group(1)) + 1
                output.append(
                    f"<h{level}>{heading.group(2)}</h{level}>"
                )

            elif bullet:
                if not in_list:
                    output.append("<ul>")
                    in_list = True

                output.append(f"<li>{bullet.group(1)}</li>")

            else:
                if in_list:
                    output.append("</ul>")
                    in_list = False

                output.append(f"<p>{line}</p>")

        if in_list:
            output.append("</ul>")

        return "\n".join(output)

    def render_value(value):
        if value is None:
            return ""

        if isinstance(value, str):
            return safe_markdown(value)

        if isinstance(value, (int, float, bool)):
            return html.escape(str(value))

        if isinstance(value, list):
            if not value:
                return ""

            if all(isinstance(item, dict) for item in value):
                return "".join(render_object(item) for item in value)

            return (
                "<ul>"
                + "".join(
                    f"<li>{render_value(item)}</li>"
                    for item in value
                )
                + "</ul>"
            )

        if isinstance(value, dict):
            return render_object(value)

        return safe_markdown(str(value))

    def render_object(obj):
        if not isinstance(obj, dict):
            return (
                "<article class='item'>"
                + render_value(obj)
                + "</article>"
            )

        title = (
            obj.get("title")
            or obj.get("concept")
            or obj.get("topic")
            or obj.get("question")
            or obj.get("activity")
            or obj.get("area")
            or obj.get("stage")
            or ""
        )

        parts = []

        if title:
            parts.append(
                f"<h3>{html.escape(str(title))}</h3>"
            )

        excluded = {
            "title", "concept", "topic", "question",
            "activity", "area", "stage"
        }

        for key, value in obj.items():
            if key in excluded or value is None or value == "":
                continue

            label = html.escape(
                str(key).replace("_", " ").title()
            )

            if isinstance(value, (list, dict)):
                if value:
                    parts.append(
                        f"<p><strong>{label}</strong></p>"
                        f"{render_value(value)}"
                    )
            else:
                parts.append(
                    f"<p><strong>{label}:</strong> "
                    f"{render_value(value)}</p>"
                )

        return (
            "<article class='item'>"
            + "".join(parts)
            + "</article>"
        )

    def render_section(title, content, section_id=None):
        body = render_value(content)

        if not body:
            return ""

        section_id_attr = (
            f" id='{html.escape(section_id, quote=True)}'"
            if section_id else ""
        )

        return (
            f"<section class='section'{section_id_attr}>"
            f"<h2>{html.escape(title)}</h2>"
            f"{body}</section>"
        )

    # ---------------------------------------------------------
    # 3. Process each course and module
    # ---------------------------------------------------------
    for module_id, course_id in zip(module_ids, course_ids):
        course_id = int(course_id)
        module_id = int(module_id)

        print(f"\nProcessing Course {course_id}, Module {module_id}")

        try:
            # -------------------------------------------------
            # 4. Get student performance
            # -------------------------------------------------
            performance = summarize_student_module_performance(
                db_config=db_config,
                course_id=course_id,
                module_id=module_id,
                student_id=student_id,
                llm=llm
            )

            if not isinstance(performance, dict):
                raise TypeError(
                    "Performance function must return a dictionary."
                )

            quiz_ids = list(dict.fromkeys(
                int(item["quiz_id"])
                for item in performance.get("quiz_results", [])
                if item.get("quiz_id") is not None
            ))

            if not quiz_ids:
                results.append({
                    "course_id": course_id,
                    "module_id": module_id,
                    "lesson_id": None,
                    "status": "skipped",
                    "html_path": None,
                    "error": "No quizzes found in the performance report."
                })
                continue

            # -------------------------------------------------
            # 5. Map quizzes to lessons
            # -------------------------------------------------
            connection = mysql.connector.connect(**db_config)

            try:
                placeholders = ", ".join(
                    ["%s"] * len(quiz_ids)
                )

                query = f"""
                    SELECT DISTINCT
                        course_id,
                        module_id,
                        quiz_id,
                        lesson_id
                    FROM tbl_course_content
                    WHERE course_id = %s
                      AND module_id = %s
                      AND quiz_id IN ({placeholders})
                      AND lesson_id IS NOT NULL
                      AND (is_deleted = 0 OR is_deleted IS NULL)
                    ORDER BY lesson_id
                """

                with connection.cursor(dictionary=True) as cursor:
                    cursor.execute(
                        query,
                        (course_id, module_id, *quiz_ids)
                    )
                    mappings = cursor.fetchall()
            finally:
                connection.close()

            quiz_to_lessons = {}

            for mapping in mappings:
                quiz_to_lessons.setdefault(
                    int(mapping["quiz_id"]), set()
                ).add(int(mapping["lesson_id"]))

            lesson_ids = sorted({
                lesson_id
                for lesson_set in quiz_to_lessons.values()
                for lesson_id in lesson_set
            })

            print("Quiz IDs:", quiz_ids)
            print("Lesson IDs:", lesson_ids)

            if not lesson_ids:
                results.append({
                    "course_id": course_id,
                    "module_id": module_id,
                    "lesson_id": None,
                    "status": "skipped",
                    "html_path": None,
                    "error": (
                        "No lessons are mapped to the module's quizzes."
                    )
                })
                continue

            # -------------------------------------------------
            # 6. Get lesson summaries
            # -------------------------------------------------
            lesson_summaries = get_lesson_summaries(
                module_ids=[module_id] * len(lesson_ids),
                lesson_ids=lesson_ids,
                course_ids=[course_id] * len(lesson_ids),
                db_config=db_config
            )

            summary_by_lesson = {
                int(item["lesson_id"]): item
                for item in lesson_summaries
                if item.get("lesson_id") is not None
            }

            quiz_result_by_id = {
                int(item["quiz_id"]): item
                for item in performance.get("quiz_results", [])
                if item.get("quiz_id") is not None
            }

            # -------------------------------------------------
            # 7. Generate a separate guide for each lesson
            # -------------------------------------------------
            for lesson_id in lesson_ids:
                try:
                    lesson_item = summary_by_lesson.get(lesson_id)

                    if not lesson_item:
                        raise ValueError(
                            "get_lesson_summaries() returned no record "
                            f"for lesson {lesson_id}."
                        )

                    if (
                        lesson_item.get("status") != "success"
                        or not lesson_item.get("summary")
                    ):
                        raise ValueError(
                            lesson_item.get(
                                "error",
                                "Lesson summary extraction failed."
                            )
                        )

                    # Find quizzes associated with this lesson.
                    related_quiz_ids = [
                        quiz_id
                        for quiz_id, linked_lessons
                        in quiz_to_lessons.items()
                        if lesson_id in linked_lessons
                    ]

                    related_quiz_results = [
                        quiz_result_by_id[quiz_id]
                        for quiz_id in related_quiz_ids
                        if quiz_id in quiz_result_by_id
                    ]

                    # Identify incorrectly answered questions.
                    related_question_ids = set()

                    for quiz in related_quiz_results:
                        for attempt in quiz.get("attempts", []):
                            for answer in attempt.get(
                                "wrong_answers", []
                            ):
                                question_id = answer.get("question_id")
                                if question_id is not None:
                                    related_question_ids.add(
                                        int(question_id)
                                    )

                    # Filter knowledge gaps when question IDs exist.
                    all_gaps = performance.get("knowledge_gaps", [])
                    related_gaps = []

                    for gap in all_gaps:
                        question_id = gap.get("question_id")

                        if question_id is None:
                            related_gaps.append(gap)
                        elif int(question_id) in related_question_ids:
                            related_gaps.append(gap)

                    lesson_performance = {
                        "course_id": course_id,
                        "module_id": module_id,
                        "lesson_id": lesson_id,
                        "student_id": str(student_id),
                        "quizzes": related_quiz_results,
                        "knowledge_gaps": related_gaps,
                        "average_percentage": performance.get(
                            "average_percentage", 0
                        ),
                        "total_correct": performance.get(
                            "total_correct", 0
                        ),
                        "total_incorrect": performance.get(
                            "total_incorrect", 0
                        )
                    }

                    lesson_text = str(lesson_item["summary"])

                    source_urls = list(dict.fromkeys(
                        url for url in lesson_item.get("urls", [])
                        if isinstance(url, str)
                        and url.startswith(("https://", "http://"))
                    ))

                    # -------------------------------------------------
                    # 8. Generate personalized learning content
                    # -------------------------------------------------
                    prompt = f"""
You are an expert personalized learning assistant.

Create a complete, detailed learning guide for the student
using the supplied lesson material and assessment evidence.

Return ONLY valid JSON with this structure:

{{
  "lesson_title": "Lesson title",
  "learning_objectives": ["Objective 1"],
  "personalized_introduction": "Personalized introduction",
  "lesson_explanation": [
    {{
      "title": "Concept",
      "explanation": "Detailed explanation",
      "simple_explanation": "Beginner-friendly explanation",
      "example": "Relevant example",
      "key_points": ["Key point"]
    }}
  ],
  "knowledge_gaps": [
    {{
      "concept": "Concept",
      "why_it_matters": "Why it matters",
      "what_went_wrong": "Observed mismatch in the answer",
      "possible_reason": "Possible misconception, not a fact",
      "explanation": "Detailed corrective explanation",
      "example": "Illustrative example",
      "practice_activity": "Targeted activity"
    }}
  ],
  "revision_notes": ["Important revision note"],
  "practice_questions": [
    {{
      "question": "Question",
      "options": ["A", "B", "C", "D"],
      "correct_answer": "A",
      "explanation": "Answer explanation",
      "difficulty": "easy",
      "related_concept": "Concept"
    }}
  ],
  "study_plan": [
    {{
      "step": 1,
      "activity": "Activity",
      "estimated_minutes": 10,
      "completion_criteria": "Success criteria"
    }}
  ],
  "self_assessment": [
    {{
      "question": "Reflection question",
      "expected_understanding": "Expected understanding"
    }}
  ],
  "encouragement": "Brief closing message"
}}

Requirements:
- Base explanations on the supplied lesson material.
- Personalize using available assessment evidence.
- Do not assume why a student answered incorrectly.
- Possible reasons must be presented as hypotheses.
- Do not invent weaknesses when no quiz was attempted.
- Include 3 to 5 practice questions.
- Each practice question must have exactly one correct answer.
- Provide actionable revision activities.
- Keep the content relevant to this lesson.
- Use empty strings or arrays when evidence is insufficient.
- Return complete JSON without Markdown fences.

COURSE: {course_id}
MODULE: {module_id}
LESSON: {lesson_id}

LESSON MATERIAL:
{lesson_text}

STUDENT PERFORMANCE:
{json.dumps(lesson_performance, ensure_ascii=False, default=str)}
"""

                    personalized = invoke_json(prompt)

                    # -------------------------------------------------
                    # 9. Render original source links
                    # -------------------------------------------------
                    source_html = ""

                    if source_urls:
                        links = []

                        for url in source_urls:
                            links.append(
                                "<li><a target='_blank' "
                                "rel='noopener noreferrer' href='"
                                + html.escape(url, quote=True)
                                + "'>Open original lesson material</a></li>"
                            )

                        source_html = (
                            "<section class='section' id='sources'>"
                            "<h2>Original lesson materials</h2>"
                            "<ul>" + "".join(links) + "</ul>"
                            "</section>"
                        )

                    # -------------------------------------------------
                    # 10. Render HTML sections
                    # -------------------------------------------------
                    sections = [
                        render_section(
                            "Learning objectives",
                            personalized.get("learning_objectives", []),
                            "objectives"
                        ),
                        render_section(
                            "Personalized introduction",
                            personalized.get(
                                "personalized_introduction", ""
                            ),
                            "introduction"
                        ),
                        render_section(
                            "Learn the concepts",
                            personalized.get("lesson_explanation", []),
                            "lesson"
                        ),
                        render_section(
                            "Concepts to improve",
                            personalized.get("knowledge_gaps", []),
                            "knowledge-gaps"
                        ),
                        render_section(
                            "Revision notes",
                            personalized.get("revision_notes", []),
                            "revision"
                        ),
                        render_section(
                            "Practice questions",
                            personalized.get("practice_questions", []),
                            "practice"
                        ),
                        render_section(
                            "Your study plan",
                            personalized.get("study_plan", []),
                            "study-plan"
                        ),
                        render_section(
                            "Self-assessment",
                            personalized.get("self_assessment", []),
                            "self-assessment"
                        ),
                        source_html,
                        render_section(
                            "Continue learning",
                            personalized.get("encouragement", ""),
                            "encouragement"
                        )
                    ]

                    title = html.escape(
                        str(
                            personalized.get("lesson_title")
                            or f"Lesson {lesson_id}"
                        )
                    )

                    safe_student_id = re.sub(
                        r"[^a-zA-Z0-9_-]",
                        "_",
                        str(student_id)
                    )

                    filename = (
                        f"course_{course_id}_module_{module_id}_"
                        f"lesson_{lesson_id}_student_{safe_student_id}.html"
                    )

                    html_path = output_dir / filename

                    # -------------------------------------------------
                    # 11. Create the complete HTML document
                    # -------------------------------------------------
                    document = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{title} - Personalized Learning</title>
<style>
* {{ box-sizing: border-box; }}
body {{
    margin: 0;
    background: #f3f6fb;
    color: #263247;
    font-family: Arial, Helvetica, sans-serif;
    line-height: 1.7;
}}
header {{
    background: #233c70;
    color: white;
    padding: 28px 20px;
}}
.container, main {{
    max-width: 1050px;
    margin: auto;
}}
header h1 {{
    margin: 0 0 8px;
    font-size: 28px;
}}
header p {{
    margin: 4px 0;
    color: #e3eaff;
}}
main {{
    padding: 20px;
}}
.section {{
    background: white;
    padding: 22px;
    margin: 20px 0;
    border-radius: 12px;
    box-shadow: 0 2px 10px rgba(0,0,0,.06);
    overflow-wrap: anywhere;
}}
.section h2 {{
    color: #233c70;
    border-bottom: 1px solid #e5eaf2;
    padding-bottom: 10px;
    margin-top: 0;
}}
.item {{
    background: #f7f9fd;
    border-left: 4px solid #6488d6;
    border-radius: 6px;
    padding: 14px 18px;
    margin: 14px 0;
}}
.item h3 {{
    color: #314e85;
    margin: 0 0 8px;
}}
li {{ margin: 6px 0; }}
a {{ color: #315fb2; }}
.meta {{
    font-size: 13px;
    color: #e3eaff;
}}
footer {{
    text-align: center;
    padding: 20px;
    font-size: 12px;
    color: #667085;
}}
@media(max-width:600px) {{
    header h1 {{ font-size: 23px; }}
    .section {{ padding: 16px; }}
}}
@media print {{
    body {{ background: white; }}
    .section {{ box-shadow: none; border: 1px solid #ddd; }}
}}
</style>
</head>
<body>
<header>
  <div class="container">
    <h1>{title}</h1>
    <p>Personalized learning guide</p>
    <p class="meta">
      Course: {course_id} |
      Module: {module_id} |
      Lesson: {lesson_id}
    </p>
    <p class="meta">
      Student: {html.escape(str(student_id))}
      | Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}
    </p>
  </div>
</header>
<main>
{''.join(section for section in sections if section)}
</main>
<footer>
Personalized learning guide generated from lesson materials
and available assessment evidence.
</footer>
</body>
</html>
"""

                    html_path.write_text(
                        document,
                        encoding="utf-8"
                    )

                    results.append({
                        "course_id": course_id,
                        "module_id": module_id,
                        "lesson_id": lesson_id,
                        "quiz_ids": related_quiz_ids,
                        "status": "success",
                        "html_path": str(html_path.resolve()),
                        "lesson_urls": source_urls,
                        "error": None
                    })

                    print(
                        f"Generated personalized lesson: {html_path}"
                    )

                except Exception as exc:
                    results.append({
                        "course_id": course_id,
                        "module_id": module_id,
                        "lesson_id": lesson_id,
                        "status": "failed",
                        "html_path": None,
                        "error": str(exc)
                    })
                    print(
                        f"Failed Course {course_id}, Module {module_id}, "
                        f"Lesson {lesson_id}: {exc}"
                    )

        except Exception as exc:
            results.append({
                "course_id": course_id,
                "module_id": module_id,
                "lesson_id": None,
                "status": "failed",
                "html_path": None,
                "error": str(exc)
            })
            print(
                f"Failed Course {course_id}, Module {module_id}: {exc}"
            )

    return results

In [82]:
results = generate_personalized_lesson_html(
    module_ids=[8],
    course_ids=[1],
    student_id=1,
    db_config=db_config,
    llm=llm
)

for item in results:
    print("\nCourse:", item["course_id"])
    print("Module:", item["module_id"])
    print("Lesson:", item["lesson_id"])
    print("Status:", item["status"])
    print("HTML:", item["html_path"])
    print("Error:", item["error"])


Processing Course 1, Module 8
Quiz IDs: [9, 4]
Lesson IDs: [11]
Requested combinations:
 module_id  lesson_id  course_id
         8         11          1

Matched combinations:
 module_id  lesson_id  course_id
         8       11.0          1
Generated personalized lesson: personalized_lessons\course_1_module_8_lesson_11_student_1.html

Course: 1
Module: 8
Lesson: 11
Status: success
HTML: C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe\mock_interviews\research\personalized_lessons\course_1_module_8_lesson_11_student_1.html
Error: None


# Focus Report

In [94]:
def generate_personalized_lesson_json(
    module_ids,
    course_ids,
    student_id,
    db_config,
    llm,
    max_retries=2
):
    import json
    import re
    import mysql.connector
    from datetime import datetime

    # ---------------------------------------------------------
    # 1. Validate inputs
    # ---------------------------------------------------------
    if len(module_ids) != len(course_ids):
        raise ValueError(
            "module_ids and course_ids must have equal lengths."
        )

    if not module_ids:
        return []

    results = []

    # ---------------------------------------------------------
    # 2. Helper functions
    # ---------------------------------------------------------
    def response_text(response):
        if isinstance(response, str):
            return response

        content = getattr(response, "content", None)

        if isinstance(content, str):
            return content

        if isinstance(content, list):
            return "\n".join(
                item if isinstance(item, str)
                else item.get("text", "")
                if isinstance(item, dict)
                else ""
                for item in content
            )

        return str(response)

    def parse_json(response):
        text = response_text(response).strip()

        text = re.sub(
            r"^\s*```(?:json)?\s*|\s*```\s*$",
            "",
            text,
            flags=re.IGNORECASE
        ).strip()

        decoder = json.JSONDecoder()

        for index, char in enumerate(text):
            if char not in "{[":
                continue

            try:
                value, _ = decoder.raw_decode(text[index:])
                if isinstance(value, dict):
                    return value
            except json.JSONDecodeError:
                continue

        raise ValueError(
            "LLM returned invalid JSON. "
            f"Response ending: {text[-500:]}"
        )

    def invoke_json(prompt):
        last_error = None

        for attempt in range(max_retries + 1):
            current_prompt = prompt

            if attempt:
                current_prompt += """

Your previous response was invalid or incomplete.
Return a shorter, complete JSON object.
Do not use Markdown fences or text outside the JSON.
Ensure all brackets and quotation marks are closed.
"""

            response = llm.invoke(current_prompt)

            try:
                return parse_json(response)
            except (ValueError, json.JSONDecodeError) as exc:
                last_error = exc
                print(
                    f"JSON parsing attempt {attempt + 1} failed: {exc}"
                )

        raise ValueError(
            f"Could not obtain valid JSON: {last_error}"
        )

    # ---------------------------------------------------------
    # 3. Process each course and module
    # ---------------------------------------------------------
    for module_id, course_id in zip(module_ids, course_ids):
        course_id = int(course_id)
        module_id = int(module_id)

        print(f"\nProcessing Course {course_id}, Module {module_id}")

        try:
            # -------------------------------------------------
            # 4. Get student performance
            # -------------------------------------------------
            performance = summarize_student_module_performance(
                db_config=db_config,
                course_id=course_id,
                module_id=module_id,
                student_id=student_id,
                llm=llm
            )

            if not isinstance(performance, dict):
                raise TypeError(
                    "Performance function must return a dictionary."
                )

            quiz_ids = list(dict.fromkeys(
                int(item["quiz_id"])
                for item in performance.get("quiz_results", [])
                if item.get("quiz_id") is not None
            ))

            if not quiz_ids:
                results.append({
                    "course_id": course_id,
                    "module_id": module_id,
                    "lesson_id": None,
                    "status": "skipped",
                    "data": None,
                    "error": "No quizzes found in the performance report."
                })
                continue

            # -------------------------------------------------
            # 5. Map quizzes to lessons
            # -------------------------------------------------
            connection = mysql.connector.connect(**db_config)

            try:
                placeholders = ", ".join(
                    ["%s"] * len(quiz_ids)
                )

                query = f"""
                    SELECT DISTINCT
                        course_id,
                        module_id,
                        quiz_id,
                        lesson_id
                    FROM tbl_course_content
                    WHERE course_id = %s
                      AND module_id = %s
                      AND quiz_id IN ({placeholders})
                      AND lesson_id IS NOT NULL
                      AND (is_deleted = 0 OR is_deleted IS NULL)
                    ORDER BY lesson_id
                """

                with connection.cursor(dictionary=True) as cursor:
                    cursor.execute(
                        query,
                        (course_id, module_id, *quiz_ids)
                    )
                    mappings = cursor.fetchall()

            finally:
                connection.close()

            quiz_to_lessons = {}

            for mapping in mappings:
                quiz_to_lessons.setdefault(
                    int(mapping["quiz_id"]), set()
                ).add(int(mapping["lesson_id"]))

            lesson_ids = sorted({
                lesson_id
                for lesson_set in quiz_to_lessons.values()
                for lesson_id in lesson_set
            })

            print("Quiz IDs:", quiz_ids)
            print("Lesson IDs:", lesson_ids)

            if not lesson_ids:
                results.append({
                    "course_id": course_id,
                    "module_id": module_id,
                    "lesson_id": None,
                    "status": "skipped",
                    "data": None,
                    "error": (
                        "No lessons are mapped to the module's quizzes."
                    )
                })
                continue

            # -------------------------------------------------
            # 6. Get lesson summaries
            # -------------------------------------------------
            lesson_summaries = get_lesson_summaries(
                module_ids=[module_id] * len(lesson_ids),
                lesson_ids=lesson_ids,
                course_ids=[course_id] * len(lesson_ids),
                db_config=db_config
            )

            summary_by_lesson = {
                int(item["lesson_id"]): item
                for item in lesson_summaries
                if item.get("lesson_id") is not None
            }

            quiz_result_by_id = {
                int(item["quiz_id"]): item
                for item in performance.get("quiz_results", [])
                if item.get("quiz_id") is not None
            }

            # -------------------------------------------------
            # 7. Generate a separate guide for each lesson
            # -------------------------------------------------
            for lesson_id in lesson_ids:
                try:
                    lesson_item = summary_by_lesson.get(lesson_id)

                    if not lesson_item:
                        raise ValueError(
                            "get_lesson_summaries() returned no record "
                            f"for lesson {lesson_id}."
                        )

                    if (
                        lesson_item.get("status") != "success"
                        or not lesson_item.get("summary")
                    ):
                        raise ValueError(
                            lesson_item.get(
                                "error",
                                "Lesson summary extraction failed."
                            )
                        )

                    # Find quizzes associated with this lesson.
                    related_quiz_ids = [
                        quiz_id
                        for quiz_id, linked_lessons
                        in quiz_to_lessons.items()
                        if lesson_id in linked_lessons
                    ]

                    related_quiz_results = [
                        quiz_result_by_id[quiz_id]
                        for quiz_id in related_quiz_ids
                        if quiz_id in quiz_result_by_id
                    ]

                    # Identify incorrectly answered questions.
                    related_question_ids = set()

                    for quiz in related_quiz_results:
                        for attempt in quiz.get("attempts", []):
                            for answer in attempt.get(
                                "wrong_answers", []
                            ):
                                question_id = answer.get("question_id")
                                if question_id is not None:
                                    related_question_ids.add(
                                        int(question_id)
                                    )

                    # Filter knowledge gaps when question IDs exist.
                    all_gaps = performance.get("knowledge_gaps", [])
                    related_gaps = []

                    for gap in all_gaps:
                        question_id = gap.get("question_id")

                        if question_id is None:
                            related_gaps.append(gap)
                        elif int(question_id) in related_question_ids:
                            related_gaps.append(gap)

                    lesson_performance = {
                        "course_id": course_id,
                        "module_id": module_id,
                        "lesson_id": lesson_id,
                        "student_id": str(student_id),
                        "quizzes": related_quiz_results,
                        "knowledge_gaps": related_gaps,
                        "average_percentage": performance.get(
                            "average_percentage", 0
                        ),
                        "total_correct": performance.get(
                            "total_correct", 0
                        ),
                        "total_incorrect": performance.get(
                            "total_incorrect", 0
                        )
                    }

                    lesson_text = str(lesson_item["summary"])

                    source_urls = list(dict.fromkeys(
                        url for url in lesson_item.get("urls", [])
                        if isinstance(url, str)
                        and url.startswith(("https://", "http://"))
                    ))

                    # -------------------------------------------------
                    # 8. Generate personalized learning content
                    # -------------------------------------------------
                    prompt = f"""
You are an expert personalized learning assistant.

Create a complete, detailed learning guide for the student
using the supplied lesson material and assessment evidence.

Return ONLY valid JSON with this structure:

{{
  "lesson_title": "Lesson title",
  "learning_objectives": ["Objective 1"],
  "personalized_introduction": "Personalized introduction",
  "lesson_explanation": [
    {{
      "title": "Concept",
      "explanation": "Detailed explanation",
      "simple_explanation": "Beginner-friendly explanation",
      "example": "Relevant example",
      "key_points": ["Key point"]
    }}
  ],
  "knowledge_gaps": [
    {{
      "concept": "Concept",
      "why_it_matters": "Why it matters",
      "what_went_wrong": "Observed mismatch in the answer",
      "possible_reason": "Possible misconception, not a fact",
      "explanation": "Detailed corrective explanation",
      "example": "Illustrative example",
      "practice_activity": "Targeted activity"
    }}
  ],
  "revision_notes": ["Important revision note"],
  "practice_questions": [
    {{
      "question": "Question",
      "options": ["A", "B", "C", "D"],
      "correct_answer": "A",
      "explanation": "Answer explanation",
      "difficulty": "easy",
      "related_concept": "Concept"
    }}
  ],
  "study_plan": [
    {{
      "step": 1,
      "activity": "Activity",
      "estimated_minutes": 10,
      "completion_criteria": "Success criteria"
    }}
  ],
  "self_assessment": [
    {{
      "question": "Reflection question",
      "expected_understanding": "Expected understanding"
    }}
  ],
  "encouragement": "Brief closing message"
}}

Requirements:
- Base explanations on the supplied lesson material.
- Personalize using available assessment evidence.
- Do not assume why a student answered incorrectly.
- Possible reasons must be presented as hypotheses.
- Do not invent weaknesses when no quiz was attempted.
- Include 3 to 5 practice questions.
- Each practice question must have exactly one correct answer.
- Provide actionable revision activities.
- Keep the content relevant to this lesson.
- Use empty strings or arrays when evidence is insufficient.
- Return complete JSON without Markdown fences.

COURSE: {course_id}
MODULE: {module_id}
LESSON: {lesson_id}

LESSON MATERIAL:
{lesson_text}

STUDENT PERFORMANCE:
{json.dumps(lesson_performance, ensure_ascii=False, default=str)}
"""

                    personalized = invoke_json(prompt)

                    # -------------------------------------------------
                    # 9. Return JSON data without saving to a file
                    # -------------------------------------------------
                    document = {
                        "course_id": course_id,
                        "module_id": module_id,
                        "lesson_id": lesson_id,
                        "student_id": str(student_id),
                        "generated_at": datetime.now().isoformat(
                            timespec="seconds"
                        ),
                        "lesson_urls": source_urls,
                        "personalized_lesson": personalized
                    }

                    results.append({
                        "course_id": course_id,
                        "module_id": module_id,
                        "lesson_id": lesson_id,
                        "quiz_ids": related_quiz_ids,
                        "status": "success",
                        "data": document,
                        "error": None
                    })

                    print(
                        f"Generated personalized lesson for "
                        f"lesson {lesson_id}"
                    )

                except Exception as exc:
                    results.append({
                        "course_id": course_id,
                        "module_id": module_id,
                        "lesson_id": lesson_id,
                        "status": "failed",
                        "data": None,
                        "error": str(exc)
                    })

                    print(
                        f"Failed Course {course_id}, Module {module_id}, "
                        f"Lesson {lesson_id}: {exc}"
                    )

        except Exception as exc:
            results.append({
                "course_id": course_id,
                "module_id": module_id,
                "lesson_id": None,
                "status": "failed",
                "data": None,
                "error": str(exc)
            })

            print(
                f"Failed Course {course_id}, Module {module_id}: {exc}"
            )

    return results

In [100]:
results = generate_personalized_lesson_json(
    module_ids=[8],
    course_ids=[5],
    student_id="ITDA1282",
    db_config=db_config,
    llm=llm
)


Processing Course 5, Module 8


In [101]:
results

[{'course_id': 5,
  'module_id': 8,
  'lesson_id': None,
  'status': 'skipped',
  'data': None,
  'error': 'No quizzes found in the performance report.'}]

In [83]:

def generate_student_focus_report(
    module_ids,
    course_ids,
    student_id,
    db_config,
    llm,
    words_per_question=100,
    min_words=85,
    max_words=115
):
    """
    Generate a personalized text report for each student and module.

    For every wrong question, generate a focused explanation based
    on the associated lesson summary.

    Returns:
        A single readable string containing the student summary
        and approximately 100 words of study material per wrong question.
    """
    import re
    import json
    import mysql.connector

    if len(module_ids) != len(course_ids):
        raise ValueError(
            "module_ids and course_ids must have equal lengths."
        )

    if not module_ids:
        return "No modules were provided."

    if words_per_question <= 0:
        raise ValueError("words_per_question must be positive.")

    def get_response_text(response):
        if isinstance(response, str):
            return response

        content = getattr(response, "content", None)
        if isinstance(content, str):
            return content

        return str(response)

    def word_count(text):
        return len(re.findall(r"\b[\w'-]+\b", text))

    def clean_text(text):
        if text is None:
            return ""
        return re.sub(r"\s+", " ", str(text)).strip()

    def get_question_text(item):
        for key in (
            "question",
            "question_text",
            "question_title",
            "text"
        ):
            if item.get(key):
                return clean_text(item[key])
        return ""

    def get_wrong_answers(quiz_results):
        """
        Collect wrong answers from the available quiz result structure.
        """
        wrong_questions = []
        seen = set()

        for quiz in quiz_results:
            quiz_id = quiz.get("quiz_id")

            attempts = quiz.get("attempts", [])

            # Support quiz-level wrong_answers as well.
            if not attempts and quiz.get("wrong_answers"):
                attempts = [{
                    "wrong_answers": quiz["wrong_answers"]
                }]

            for attempt in attempts:
                wrong_answers = attempt.get("wrong_answers", [])

                for question in wrong_answers:
                    if not isinstance(question, dict):
                        question = {"question": str(question)}

                    question_text = get_question_text(question)
                    question_id = question.get("question_id")

                    identity = (
                        str(quiz_id),
                        str(question_id),
                        question_text
                    )

                    if identity in seen:
                        continue

                    seen.add(identity)

                    wrong_questions.append({
                        "quiz_id": quiz_id,
                        "question_id": question_id,
                        "question": question_text,
                        "student_answer": (
                            question.get("student_answer")
                            or question.get("selected_answer")
                            or question.get("user_answer")
                            or question.get("answer")
                            or ""
                        ),
                        "correct_answer": (
                            question.get("correct_answer")
                            or question.get("expected_answer")
                            or ""
                        ),
                        "explanation": (
                            question.get("explanation")
                            or question.get("feedback")
                            or ""
                        ),
                        "raw": question
                    })

        return wrong_questions

    def generate_focus_text(
        question,
        lesson_summary,
        target_words
    ):
        question_text = question["question"] or (
            "Question ID: " + str(question["question_id"])
        )

        prompt = f"""
You are a personalized learning assistant.

Write a focused study explanation for a student who answered
the following question incorrectly.

The explanation must be approximately {target_words} words.
Aim for {min_words} to {max_words} words.

Use only the supplied lesson material for factual explanations.
Do not introduce unrelated concepts or unsupported facts.
Do not invent the student's reasoning or misconception.

Include:
1. The concept the student should focus on.
2. A clear explanation of that concept.
3. A relevant detail or example from the lesson material.
4. A short recommendation for revision.

If a correct answer is provided, explain why it is correct
using the lesson material. If no correct answer is provided,
do not invent one.

Return plain text only. Do not return JSON, HTML or Markdown
headings. Keep the explanation focused on this question.

QUESTION:
{question_text}

STUDENT'S ANSWER:
{question["student_answer"] or "Not available"}

CORRECT ANSWER:
{question["correct_answer"] or "Not available"}

EXISTING FEEDBACK:
{question["explanation"] or "Not available"}

LESSON MATERIAL:
{lesson_summary}
"""

        last_text = ""

        for attempt in range(3):
            current_prompt = prompt

            if attempt > 0:
                current_prompt += f"""

Your previous response contained {word_count(last_text)} words.
Rewrite it to contain between {min_words} and {max_words} words.
Return only the revised explanation.
"""

            response = llm.invoke(current_prompt)
            last_text = clean_text(get_response_text(response))

            if min_words <= word_count(last_text) <= max_words:
                return last_text

        # Preserve the generated text if the model does not
        # consistently follow the word-count instruction.
        return last_text

    reports = []

    for module_id, course_id in zip(module_ids, course_ids):
        course_id = int(course_id)
        module_id = int(module_id)

        report_parts = [
            f"STUDENT LEARNING REPORT",
            f"Student ID: {student_id}",
            f"Course ID: {course_id}",
            f"Module ID: {module_id}",
            ""
        ]

        try:
            # 1. Retrieve student performance.
            performance = summarize_student_module_performance(
                db_config=db_config,
                course_id=course_id,
                module_id=module_id,
                student_id=student_id,
                llm=llm
            )

            if not isinstance(performance, dict):
                raise TypeError(
                    "Performance function must return a dictionary."
                )

            quiz_results = performance.get("quiz_results", [])
            wrong_questions = get_wrong_answers(quiz_results)

            # 2. Build the quiz-to-lesson mapping.
            quiz_ids = list(dict.fromkeys(
                int(item["quiz_id"])
                for item in quiz_results
                if item.get("quiz_id") is not None
            ))

            quiz_to_lessons = {}

            if quiz_ids:
                connection = mysql.connector.connect(**db_config)

                try:
                    placeholders = ", ".join(
                        ["%s"] * len(quiz_ids)
                    )

                    query = f"""
                        SELECT DISTINCT
                            quiz_id,
                            lesson_id
                        FROM tbl_course_content
                        WHERE course_id = %s
                          AND module_id = %s
                          AND quiz_id IN ({placeholders})
                          AND lesson_id IS NOT NULL
                          AND (is_deleted = 0 OR is_deleted IS NULL)
                    """

                    with connection.cursor(dictionary=True) as cursor:
                        cursor.execute(
                            query,
                            (course_id, module_id, *quiz_ids)
                        )
                        mappings = cursor.fetchall()

                    for mapping in mappings:
                        quiz_id = int(mapping["quiz_id"])
                        lesson_id = int(mapping["lesson_id"])

                        quiz_to_lessons.setdefault(
                            quiz_id, set()
                        ).add(lesson_id)

                finally:
                    connection.close()

            # 3. Identify lessons for the wrong questions.
            lesson_ids = sorted({
                lesson_id
                for question in wrong_questions
                for lesson_id in quiz_to_lessons.get(
                    int(question["quiz_id"]), set()
                )
            })

            summaries = []

            if lesson_ids:
                summaries = get_lesson_summaries(
                    module_ids=[module_id] * len(lesson_ids),
                    lesson_ids=lesson_ids,
                    course_ids=[course_id] * len(lesson_ids),
                    db_config=db_config
                )

            summary_by_lesson = {
                int(item["lesson_id"]): item
                for item in summaries
                if item.get("lesson_id") is not None
            }

            # 4. Add the student-level summary.
            total_correct = performance.get("total_correct", 0)
            total_incorrect = performance.get("total_incorrect", 0)
            average = performance.get("average_percentage")

            report_parts.append("STUDENT SUMMARY")
            report_parts.append(
                f"Total correct answers: {total_correct}"
            )
            report_parts.append(
                f"Total incorrect answers: {total_incorrect}"
            )

            if average is not None:
                report_parts.append(
                    f"Average score: {average}%"
                )

            report_parts.append(
                f"Questions requiring review: {len(wrong_questions)}"
            )
            report_parts.append("")

            # 5. Generate an individual focus explanation
            # for every incorrect question.
            report_parts.append("CONTENT TO FOCUS ON")

            if not wrong_questions:
                report_parts.append(
                    "No incorrect answers were found in the "
                    "available quiz results."
                )
            else:
                for index, question in enumerate(
                    wrong_questions, start=1
                ):
                    question_text = (
                        question["question"]
                        or f"Question ID {question['question_id']}"
                    )

                    related_lessons = sorted(
                        quiz_to_lessons.get(
                            int(question["quiz_id"]), set()
                        )
                    )

                    available_summaries = [
                        summary_by_lesson[lesson_id]
                        for lesson_id in related_lessons
                        if lesson_id in summary_by_lesson
                        and summary_by_lesson[lesson_id].get("status")
                        == "success"
                        and summary_by_lesson[lesson_id].get("summary")
                    ]

                    report_parts.append("")
                    report_parts.append(
                        f"{index}. Question: {question_text}"
                    )
                    report_parts.append(
                        f"Quiz ID: {question['quiz_id']}"
                    )

                    if not available_summaries:
                        report_parts.append(
                            "Focus explanation unavailable: "
                            "no successfully extracted lesson "
                            "material was found for this question."
                        )
                        continue

                    lesson_material = "\n\n".join(
                        str(item["summary"])
                        for item in available_summaries
                    )

                    focus_text = generate_focus_text(
                        question,
                        lesson_material,
                        words_per_question
                    )

                    report_parts.append(
                        f"Study focus ({word_count(focus_text)} words):"
                    )
                    report_parts.append(focus_text)

            reports.append("\n".join(report_parts))

        except Exception as exc:
            reports.append(
                "\n".join([
                    "STUDENT LEARNING REPORT",
                    f"Student ID: {student_id}",
                    f"Course ID: {course_id}",
                    f"Module ID: {module_id}",
                    "Status: Failed",
                    f"Error: {exc}"
                ])
            )

    return "\n\n" + ("\n\n" + "=" * 70 + "\n\n").join(reports)

In [89]:
import mysql.connector
from mysql.connector import Error

def get_wrong_answers(db_config, student_id, quiz_id):
    """
    Retrieve all wrong answers for a student in a particular quiz.

    Args:
        db_config (dict): MySQL database configuration.
        student_id (str): Student ID.
        quiz_id (int): Quiz ID.

    Returns:
        list[dict]: List of wrong answers with question details
                    and selected answers.
    """
    connection = None

    try:
        connection = mysql.connector.connect(**db_config)

        query = """
            SELECT
                qa.attempt_id,
                a.question_id,
                q.question_number,
                q.question_text,
                q.options_json,
                a.selected_answer,
                q.correct_answer,
                q.difficulty,
                q.bloom_level
            FROM datapro_lms.tbl_quiz_attempts AS qa
            INNER JOIN datapro_lms.tbl_quiz_attempt_answers AS a
                ON qa.attempt_id = a.attempt_id
            INNER JOIN datapro_lms.tbl_quiz_questions AS q
                ON a.question_id = q.question_id
            WHERE qa.student_id = %s
              AND qa.quiz_id = %s
              AND a.is_correct = 0
            ORDER BY qa.submitted_at DESC, q.question_number ASC
        """

        with connection.cursor(dictionary=True) as cursor:
            cursor.execute(query, (student_id, quiz_id))
            results = cursor.fetchall()

        return results

    except Error as e:
        print(f"Database error: {e}")
        raise

    finally:
        if connection and connection.is_connected():
            connection.close()

In [90]:

from pathlib import Path
import re
import mysql.connector
from mysql.connector import Error


def generate_student_focus_report(
    module_ids,
    course_ids,
    student_id,
    db_config,
    llm,
    output_dir="student_reports",
    save_report=True,
    words_per_question=100,
):
    """
    Generate a text-based learning report for a student.

    Workflow:
      1. Get student performance summary.
      2. Retrieve incorrect answers using get_wrong_answers().
      3. Map each quiz to its lesson.
      4. Extract and summarize lesson files using
         get_lesson_summaries().
      5. Generate a focused explanation for each wrong answer.

    Returns:
        str: Plain-text student learning report.
    """

    if len(module_ids) != len(course_ids):
        raise ValueError(
            "module_ids and course_ids must have equal lengths."
        )

    if not module_ids:
        raise ValueError("At least one module is required.")

    def get_lesson_ids(course_id, module_id, quiz_id):
        """Find lesson IDs associated with a quiz."""

        connection = mysql.connector.connect(**db_config)

        try:
            with connection.cursor(dictionary=True) as cursor:
                cursor.execute(
                    """
                    SELECT DISTINCT lesson_id
                    FROM datapro_lms.tbl_course_content
                    WHERE course_id = %s
                      AND module_id = %s
                      AND quiz_id = %s
                      AND (is_deleted = 0 OR is_deleted IS NULL)
                    """,
                    (course_id, module_id, quiz_id),
                )
                rows = cursor.fetchall()

            return [
                int(row["lesson_id"])
                for row in rows
                if row.get("lesson_id") is not None
            ]

        finally:
            connection.close()

    def get_question_text(row):
        return str(row.get("question_text") or "").strip()

    def generate_focus_explanation(question, correct_answer,
                                   selected_answer, lesson_content):
        """Generate an explanation grounded in the lesson material."""

        prompt = f"""
You are an educational tutor generating a personalized
learning report.

Analyze the following incorrectly answered question using
ONLY the supplied lesson content.

QUESTION:
{question}

STUDENT'S SELECTED ANSWER:
{selected_answer}

CORRECT ANSWER:
{correct_answer}

LESSON CONTENT:
{lesson_content}

TASK:
Write approximately {words_per_question} words of focused
learning content to help the student understand this question.

INSTRUCTIONS:
1. Explain the relevant concept from the lesson.
2. Explain the correct answer using the lesson content.
3. Clarify the relevant distinctions or steps, if present.
4. Focus on what the student needs to review.
5. Do not invent information absent from the lesson.
6. Do not assume why the student selected the wrong answer.
7. Do not merely repeat the question and answer.
8. Return plain text only.
9. Aim for approximately {words_per_question} words.
"""

        response = llm.invoke(prompt)

        if hasattr(response, "content"):
            return str(response.content).strip()

        return str(response).strip()

    report_sections = []

    for module_id, course_id in zip(module_ids, course_ids):

        # 1. Get the existing student performance summary.
        performance = summarize_student_module_performance(
            db_config=db_config,
            course_id=course_id,
            module_id=module_id,
            student_id=student_id,
            llm=llm,
        )

        if not isinstance(performance, dict):
            raise TypeError(
                "Performance function must return a dictionary."
            )

        # 2. Retrieve quiz IDs and incorrect questions.
        quiz_results = performance.get("quiz_results", [])

        if isinstance(quiz_results, dict):
            quiz_results = list(quiz_results.values())

        quiz_ids = list(dict.fromkeys(
            int(q["quiz_id"])
            for q in quiz_results
            if q.get("quiz_id") is not None
            and str(q["quiz_id"]).isdigit()
        ))

        questions_by_quiz = {}

        for quiz_id in quiz_ids:
            wrong_questions = get_wrong_answers(
                db_config=db_config,
                student_id=student_id,
                quiz_id=quiz_id,
            )

            if wrong_questions:
                questions_by_quiz[quiz_id] = wrong_questions

        # 3. Map each quiz to its lesson IDs.
        quiz_lesson_map = {}
        all_lesson_ids = []

        for quiz_id in questions_by_quiz:
            lesson_ids = get_lesson_ids(
                course_id=course_id,
                module_id=module_id,
                quiz_id=quiz_id,
            )

            quiz_lesson_map[quiz_id] = lesson_ids
            all_lesson_ids.extend(lesson_ids)

        all_lesson_ids = list(dict.fromkeys(all_lesson_ids))

        # 4. Extract lesson files and generate lesson summaries
        #    using the existing, working function.
        summary_lookup = {}

        if all_lesson_ids:
            summaries = get_lesson_summaries(
                module_ids=[module_id] * len(all_lesson_ids),
                lesson_ids=all_lesson_ids,
                course_ids=[course_id] * len(all_lesson_ids),
                db_config=db_config,
            )

            for item in summaries or []:
                if not isinstance(item, dict):
                    continue

                if str(item.get("status", "")).lower() not in (
                    "success", "completed", "ok"
                ):
                    continue

                key = (
                    str(item.get("course_id")),
                    str(item.get("module_id")),
                    str(item.get("lesson_id")),
                )

                summary = str(item.get("summary") or "").strip()

                if summary:
                    summary_lookup[key] = summary

        # 5. Build the report.
        total_correct = performance.get("total_correct", 0)
        total_incorrect = performance.get(
            "total_incorrect",
            sum(len(qs) for qs in questions_by_quiz.values()),
        )
        average_score = performance.get(
            "average_percentage",
            performance.get("average_score", 0),
        )

        all_wrong_questions = [
            (quiz_id, question)
            for quiz_id, questions in questions_by_quiz.items()
            for question in questions
        ]

        section = [
            "STUDENT LEARNING REPORT",
            f"Student ID: {student_id}",
            f"Course ID: {course_id}",
            f"Module ID: {module_id}",
            "",
            "STUDENT SUMMARY",
            f"Total correct answers: {total_correct}",
            f"Total incorrect answers: {total_incorrect}",
            f"Average score: {average_score}%",
            f"Questions requiring review: {len(all_wrong_questions)}",
            "",
            "CONTENT TO FOCUS ON",
        ]

        if not all_wrong_questions:
            section.append(
                "No incorrect answers were found for the available quizzes."
            )

        for index, (quiz_id, question_data) in enumerate(
            all_wrong_questions, start=1
        ):
            question = get_question_text(question_data)
            correct_answer = str(
                question_data.get("correct_answer") or ""
            )
            selected_answer = str(
                question_data.get("selected_answer") or ""
            )

            section.extend([
                "",
                f"{index}. Question: {question}",
                f"Quiz ID: {quiz_id}",
            ])

            # Find the lesson material for this question's quiz.
            lesson_ids = quiz_lesson_map.get(quiz_id, [])

            lesson_material = []

            for lesson_id in lesson_ids:
                key = (
                    str(course_id),
                    str(module_id),
                    str(lesson_id),
                )
                content = summary_lookup.get(key, "")

                if content:
                    lesson_material.append(
                        f"Lesson {lesson_id}:\n{content}"
                    )

            if not lesson_material:
                section.append(
                    "Focus explanation unavailable: no successfully "
                    "extracted lesson summary was found for this quiz."
                )
                continue

            try:
                explanation = generate_focus_explanation(
                    question=question,
                    correct_answer=correct_answer,
                    selected_answer=selected_answer,
                    lesson_content="\n\n".join(lesson_material),
                )

                section.extend([
                    "CONTENT TO REVIEW:",
                    explanation,
                ])

            except Exception as exc:
                section.append(
                    f"Explanation generation failed: {exc}"
                )

        report_sections.append("\n".join(section))

    report = "\n\n" + (
        "\n\n" + "=" * 70 + "\n\n"
    ).join(report_sections)

    # 6. Save the text report if requested.
    if save_report:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        safe_id = re.sub(r"[^A-Za-z0-9_-]", "_", str(student_id))
        filename = f"student_{safe_id}_focus_report.txt"
        file_path = output_path / filename

        file_path.write_text(report, encoding="utf-8")
        print(f"Report saved to: {file_path.resolve()}")

    return report

In [102]:
report = generate_student_focus_report(
    module_ids=[11],
    course_ids=[1],
    student_id="ITDA1282",
    db_config=db_config,
    llm=llm,
    words_per_question=100,
    save_report=True,
)

print(report)

Report saved to: C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe\mock_interviews\research\student_reports\student_ITDA1282_focus_report.txt


STUDENT LEARNING REPORT
Student ID: ITDA1282
Course ID: 1
Module ID: 11

STUDENT SUMMARY
Total correct answers: 0
Total incorrect answers: 0
Average score: 0.0%
Questions requiring review: 0

CONTENT TO FOCUS ON
No incorrect answers were found for the available quizzes.
